In [ ]:
import os
import re
import json
import time
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Any

import numpy as np
import nibabel as nib
from scipy import ndimage as ndi

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

import nibabel as nib
from nibabel.processing import resample_from_to

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []


In [ ]:
def mount_drive_in_colab() -> None:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('FILL IN WITH DIRECTORY FOR COLAB DRIVE MOUNT', force_remount=False)
    except Exception as exc:
        print(f"Skipping Drive mount: {exc}")

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class AverageMeter:
    def __init__(self) -> None:
        self.reset()

    def reset(self) -> None:
        self.sum = 0.0
        self.count = 0
        self.avg = 0.0

    def update(self, value: float, n: int = 1) -> None:
        self.sum += value * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def save_json(obj: Dict, path: str) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2)


def load_json(path: str) -> Dict:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def is_zone_identifier(path: Path) -> bool:
    return ':Zone.Identifier' in path.name


def case_sort_key(case_id: str) -> Tuple:
    nums = re.findall(r'\d+', case_id)
    return tuple(int(n) for n in nums) if nums else (case_id,)


def count_parameters(model: nn.Module) -> Tuple[int, int]:
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params


def print_parameter_summary(
    model: nn.Module,
    model_name: str = "acf_net-base",
    baseline_trainable_m: float = 30.43,
    print_modules: bool = True,
) -> None:
    total_params, trainable_params = count_parameters(model)
    total_m = total_params / 1e6
    trainable_m = trainable_params / 1e6
    diff_m = trainable_m - baseline_trainable_m

    print("\n" + "=" * 80)
    print(f"{model_name} parameter summary")
    print("=" * 80)
    print(f"Total parameters:      {total_m:.2f}M")
    print(f"Trainable parameters:  {trainable_m:.2f}M")
    print(f"Baseline reference model trainable parameters: {baseline_trainable_m:.2f}M")
    print(f"Difference from baseline: {diff_m:+.2f}M")

    if total_params != trainable_params:
        print(f"Frozen/non-trainable parameters: {(total_params - trainable_params) / 1e6:.2f}M")

    if print_modules:
        print("\nTop-level module parameter counts:")
        for name, module in model.named_children():
            params = sum(p.numel() for p in module.parameters())
            print(f"  {name:<28} {params / 1e6:.3f}M")
    print("=" * 80)


def print_gpu_memory(label: str = "") -> None:
    if not torch.cuda.is_available():
        return
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    max_allocated = torch.cuda.max_memory_allocated() / (1024 ** 3)
    prefix = f"[GPU memory {label}]" if label else "[GPU memory]"
    print(f"{prefix} allocated={allocated:.2f} GB | reserved={reserved:.2f} GB | max_allocated={max_allocated:.2f} GB")


def print_debug_shapes_once(
    image: torch.Tensor,
    priors: torch.Tensor,
    region: torch.Tensor,
    outputs: Dict[str, torch.Tensor],
    prefix: str = "[DEBUG_SHAPES]",
) -> None:
    print("\n" + "=" * 80)
    print(prefix)
    print("=" * 80)
    print(f"image:                 {tuple(image.shape)}")
    print(f"priors:                {tuple(priors.shape)}")
    print(f"region:                {tuple(region.shape)}")
    print(f"logits_expert_1:       {tuple(outputs['logits_expert_1'].shape)}")
    print(f"logits_expert_2:       {tuple(outputs['logits_expert_2'].shape)}")
    print(f"logits_expert_3:       {tuple(outputs['logits_expert_3'].shape)}")
    print(f"region_weights:        {tuple(outputs['region_weights'].shape)}")
    print(f"router_scores:         {tuple(outputs['router_scores'].shape)}")
    print(f"image-level weights:   {tuple(outputs['weights'].shape)}")
    if "voxel_weights" in outputs:
        print(f"voxel_weights:         {tuple(outputs['voxel_weights'].shape)}")
    print(f"logits_fused:          {tuple(outputs['logits_fused'].shape)}")
    print("=" * 80 + "\n")


In [ ]:
@dataclass
class TrainConfig:
    smoke_test: bool = False
    smoke_test_epochs: int = 2
    precompute_all_priors_before_training: bool = False

    # Real dataset roots. These point to the same data as the original reference model notebook.
    brats_root: str = 'FILL IN WITH DIRECTORY FOR BRATS GLI TRAINING DATA/'
    fastsurfer_root: str = 'FILL IN WITH DIRECTORY FOR FASTSURFER TRAINING OUTPUTS/'

    # acf_net-base isolated output/cache/checkpoint paths.
    # This prevents commingling with the original reference model run files.
    work_dir: str = 'FILL IN WITH DIRECTORY FOR ACF NET BASE BRATS RUNS'
    cache_dir: str = 'FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE'
    checkpoint_dir: str = 'FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS'
    log_dir: str = 'FILL IN WITH DIRECTORY FOR TRAINING LOGS'
    manifest_path: str = 'FILL IN WITH DIRECTORY FOR RUN MANIFEST/manifest.json'
    split_path: str = 'FILL IN WITH DIRECTORY FOR TRAIN VALIDATION TEST SPLITS/splits.json'

    # Split generation
    split_seed: int = 42
    train_ratio: float = 0.6
    val_ratio: float = 0.2
    test_ratio: float = 0.2
    regenerate_manifest: bool = False
    regenerate_splits: bool = False

    # Data
    num_classes: int = 4
    batch_size: int = 2
    num_workers: int = 2
    pin_memory: bool = True
    persistent_workers: bool = True
    prefetch_factor: int = 2
    patch_size: Tuple[int, int, int] = (96, 96, 96)
    use_patch_sampling: bool = False
    cache_priors: bool = True
    debug_resampling_shapes: bool = False

    # acf_net-base router settings
    num_regions: int = 6
    region_vocab: int = 16
    router_hidden: int = 32
    router_region_dim: int = 8
    router_global_dim: int = 16
    router_calibration_lambda: float = 1.0

    # Training
    seed: int = 42
    lr: float = 2e-4
    weight_decay: float = 1e-5
    max_epochs: int = 300
    patience: int = 25
    monitor_mode: str = 'max'
    monitor_metric: str = 'val_mean_dice'
    min_delta: float = 1e-4
    amp: bool = True
    grad_clip_norm: float = 12.0
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Checkpointing / resume
    save_every: int = 1
    keep_last_k: int = 3
    resume_path: Optional[str] = None

    # Explicit ACR engineering flags
    AUTO_RESUME: bool = True
    DEBUG_SHAPES: bool = True
    SAVE_EVERY_EPOCH: bool = True
    SAVE_BEST_ONLY: bool = False


In [ ]:
BRATS_MODALITY_SUFFIXES = {
    't1': 't1n',
    't1ce': 't1c',
    't2': 't2w',
    'flair': 't2f',
}

FASTSURFER_FILES = {
    'dkt': 'aparc.DKTatlas+aseg.deep.mgz',
    'aseg': 'aseg.auto_noCCseg.mgz',
    'mask': 'mask.mgz',
}

In [ ]:
def build_case_record(case_id: str, brats_root: Path, fastsurfer_root: Path) -> Optional[Dict]:
    brats_case_dir = brats_root / case_id
    fs_case_mri_dir = fastsurfer_root / case_id / 'mri'

    if not brats_case_dir.is_dir() or not fs_case_mri_dir.is_dir():
        return None

    files = {
        't1': brats_case_dir / f'{case_id}-{BRATS_MODALITY_SUFFIXES["t1"]}.nii.gz',
        't1ce': brats_case_dir / f'{case_id}-{BRATS_MODALITY_SUFFIXES["t1ce"]}.nii.gz',
        't2': brats_case_dir / f'{case_id}-{BRATS_MODALITY_SUFFIXES["t2"]}.nii.gz',
        'flair': brats_case_dir / f'{case_id}-{BRATS_MODALITY_SUFFIXES["flair"]}.nii.gz',
        'seg': brats_case_dir / f'{case_id}-seg.nii.gz',
        'dkt': fs_case_mri_dir / FASTSURFER_FILES['dkt'],
        'aseg': fs_case_mri_dir / FASTSURFER_FILES['aseg'],
        'mask': fs_case_mri_dir / FASTSURFER_FILES['mask'],
    }

    missing = [k for k, p in files.items() if (not p.exists()) or is_zone_identifier(p)]
    if missing:
        return None

    return {
        'id': case_id,
        'brats_dir': str(brats_case_dir),
        'fastsurfer_mri_dir': str(fs_case_mri_dir),
        'mri': {
            't1': str(files['t1']),
            't1ce': str(files['t1ce']),
            't2': str(files['t2']),
            'flair': str(files['flair']),
        },
        'seg': str(files['seg']),
        'fastsurfer': {
            'dkt': str(files['dkt']),
            'aseg': str(files['aseg']),
            'mask': str(files['mask']),
        },
    }


def scan_and_match_cases(cfg: TrainConfig) -> List[Dict]:
    brats_root = Path(cfg.brats_root)
    fastsurfer_root = Path(cfg.fastsurfer_root)

    brats_case_ids = {
        p.name for p in brats_root.iterdir()
        if p.is_dir() and p.name.startswith('BraTS-GLI-') and not is_zone_identifier(p)
    }
    fs_case_ids = {
        p.name for p in fastsurfer_root.iterdir()
        if p.is_dir() and p.name.startswith('BraTS-GLI-') and not is_zone_identifier(p)
    }

    shared_case_ids = sorted(brats_case_ids.intersection(fs_case_ids), key=case_sort_key)
    manifest: List[Dict] = []

    for case_id in shared_case_ids:
        rec = build_case_record(case_id, brats_root, fastsurfer_root)
        if rec is not None:
            manifest.append(rec)

    return manifest


def load_or_create_manifest(cfg: TrainConfig) -> List[Dict]:
    ensure_dir(os.path.dirname(cfg.manifest_path))
    if os.path.exists(cfg.manifest_path) and not cfg.regenerate_manifest:
        data = load_json(cfg.manifest_path)
        return data['cases']

    cases = scan_and_match_cases(cfg)
    save_json({'cases': cases}, cfg.manifest_path)
    return cases

In [ ]:
def generate_splits_from_manifest(cfg: TrainConfig, cases: List[Dict]) -> Dict[str, List[Dict]]:
    ids = list(range(len(cases)))
    rng = random.Random(cfg.split_seed)
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(round(n * cfg.train_ratio))
    n_val = int(round(n * cfg.val_ratio))
    n_test = n - n_train - n_val

    if n_test < 1 and n >= 3:
        n_test = 1
        if n_train > n_val:
            n_train -= 1
        else:
            n_val -= 1

    train_idx = ids[:n_train]
    val_idx = ids[n_train:n_train + n_val]
    test_idx = ids[n_train + n_val:]

    splits = {
        'train': [cases[i] for i in train_idx],
        'val': [cases[i] for i in val_idx],
        'test': [cases[i] for i in test_idx],
    }
    return splits


def load_or_create_splits(cfg: TrainConfig, cases: List[Dict]) -> Dict[str, List[Dict]]:
    ensure_dir(os.path.dirname(cfg.split_path))
    if os.path.exists(cfg.split_path) and not cfg.regenerate_splits:
        return load_json(cfg.split_path)

    splits = generate_splits_from_manifest(cfg, cases)
    save_json(splits, cfg.split_path)
    return splits

In [ ]:
def load_nib(path: str) -> nib.spatialimages.SpatialImage:
    return nib.load(path)


def load_volume(path: str, dtype=np.float32) -> np.ndarray:
    img = load_nib(path)
    arr = np.asanyarray(img.dataobj)
    return arr.astype(dtype, copy=False)


def load_volume_with_meta(path: str, dtype=np.float32):
    img = load_nib(path)
    arr = np.asanyarray(img.dataobj).astype(dtype, copy=False)
    return arr, img.affine, img.header, img


def zscore_nonzero(volume: np.ndarray) -> np.ndarray:
    volume = volume.astype(np.float32)
    mask = volume != 0
    if mask.sum() == 0:
        return volume
    vals = volume[mask]
    mean = float(vals.mean())
    std = float(vals.std())
    std = max(std, 1e-6)
    out = volume.copy()
    out[mask] = (out[mask] - mean) / std
    return out


def remap_brats_labels(seg: np.ndarray) -> np.ndarray:
    seg = seg.astype(np.int64)
    # BraTS 2023 labels commonly use 0,1,2,3 already in GLI format.
    # This keeps them unchanged. If data contains 4, map 4->3.
    seg = seg.copy()
    seg[seg == 4] = 3
    return seg

In [ ]:
def _make_image_from_array(array: np.ndarray, affine: np.ndarray, header=None, is_label: bool = False):
    """
    Wrap a numpy array in a nib image for resampling.
    """
    if is_label:
        array = array.astype(np.int16, copy=False)
    else:
        array = array.astype(np.float32, copy=False)
    return nib.Nifti1Image(array, affine, header=header)


def resample_array_to_reference(
    src_array: np.ndarray,
    src_affine: np.ndarray,
    ref_img: nib.spatialimages.SpatialImage,
    order: int,
    src_header=None,
) -> np.ndarray:
    """
    Resample a single 3D array into the reference image space.
    order=1 for continuous maps
    order=0 for discrete label maps
    """
    src_img = _make_image_from_array(src_array, src_affine, header=src_header, is_label=(order == 0))
    out_img = resample_from_to(src_img, ref_img, order=order)
    out = np.asanyarray(out_img.dataobj)

    if order == 0:
        return np.rint(out).astype(np.int32)
    return out.astype(np.float32, copy=False)


def resample_priors_and_region_to_brats_space(
    priors: np.ndarray,
    region: np.ndarray,
    src_affine: np.ndarray,
    ref_img: nib.spatialimages.SpatialImage,
    src_header=None,
):
    """
    priors: [C, Z, Y, X] continuous channels
    region: [Z, Y, X] discrete coarse region map
    """
    resampled_priors = []
    for c in range(priors.shape[0]):
        resampled_c = resample_array_to_reference(
            src_array=priors[c],
            src_affine=src_affine,
            ref_img=ref_img,
            order=1,
            src_header=src_header,
        )
        resampled_priors.append(resampled_c)

    resampled_priors = np.stack(resampled_priors, axis=0).astype(np.float32, copy=False)

    resampled_region = resample_array_to_reference(
        src_array=region,
        src_affine=src_affine,
        ref_img=ref_img,
        order=0,
        src_header=src_header,
    )

    return resampled_priors, resampled_region

In [ ]:
def derive_anatomy_priors_in_fastsurfer_space(case: Dict):
    """
    Build anatomy priors and coarse region map in native FastSurfer space.

    Returns:
        priors_fs: np.ndarray [C, Z, Y, X]
        region_fs: np.ndarray [Z, Y, X]
        fs_affine: np.ndarray
        fs_header: nib header
    """
    dkt_img = load_nib(case['fastsurfer']['dkt'])
    aseg_img = load_nib(case['fastsurfer']['aseg'])
    mask_img = load_nib(case['fastsurfer']['mask'])

    dkt = np.asanyarray(dkt_img.dataobj).astype(np.int32, copy=False)
    aseg = np.asanyarray(aseg_img.dataobj).astype(np.int32, copy=False)
    mask = np.asanyarray(mask_img.dataobj).astype(np.float32, copy=False)

    # Basic tissue-like masks from aseg / dkt.
    wm = np.isin(aseg, [2, 41]).astype(np.float32)
    ventricle = np.isin(aseg, [4, 5, 14, 15, 43, 44]).astype(np.float32)
    deep_gray = np.isin(aseg, [10, 11, 12, 13, 17, 18, 26, 49, 50, 51, 52, 53, 54, 58]).astype(np.float32)
    cortex = ((dkt >= 1000) & (dkt < 4000)).astype(np.float32)
    csf = np.logical_or(ventricle > 0, np.isin(aseg, [24]).astype(bool)).astype(np.float32)

    gm = np.logical_or(cortex > 0, deep_gray > 0).astype(np.float32)

    # Restrict tissue priors to mask if desired.
    brain_mask = (mask > 0).astype(np.float32)
    wm *= brain_mask
    gm *= brain_mask
    csf *= brain_mask
    ventricle *= brain_mask
    deep_gray *= brain_mask
    cortex *= brain_mask

    # Distance maps in FastSurfer space.
    wm_dist = ndi.distance_transform_edt(1.0 - wm).astype(np.float32)
    vent_dist = ndi.distance_transform_edt(1.0 - ventricle).astype(np.float32)

    # Boundary map from anatomical segmentation.
    anat_labels = dkt.copy()
    anat_boundary = np.zeros_like(anat_labels, dtype=bool)
    anat_boundary[:-1, :, :] |= (anat_labels[:-1, :, :] != anat_labels[1:, :, :])
    anat_boundary[:, :-1, :] |= (anat_labels[:, :-1, :] != anat_labels[:, 1:, :])
    anat_boundary[:, :, :-1] |= (anat_labels[:, :, :-1] != anat_labels[:, :, 1:])
    anat_boundary = anat_boundary.astype(np.float32) * brain_mask

    # Coarse region map
    # 0 background / outside mask
    # 1 cortex
    # 2 white matter
    # 3 ventricles / CSF
    # 4 deep gray
    # 5 other brain
    region = np.zeros_like(dkt, dtype=np.int32)
    region[brain_mask > 0] = 5
    region[wm > 0] = 2
    region[gm > 0] = 1
    region[deep_gray > 0] = 4
    region[np.logical_or(ventricle > 0, csf > 0)] = 3
    region[brain_mask == 0] = 0

    priors_fs = np.stack(
        [
            wm,
            gm,
            csf,
            ventricle,
            deep_gray,
            cortex,
            wm_dist,
            vent_dist,
            anat_boundary,
        ],
        axis=0,
    ).astype(np.float32, copy=False)

    return priors_fs, region, dkt_img.affine, dkt_img.header

In [ ]:
def normalize_distance_channel(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32, copy=False)
    x_min = float(x.min())
    x_max = float(x.max())
    if x_max - x_min < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return (x - x_min) / (x_max - x_min)

In [ ]:
VENTRICLE_LABELS = {4, 5, 14, 15, 24, 31, 43, 44, 63}
DEEP_GRAY_LABELS = {10, 11, 12, 13, 17, 18, 26, 49, 50, 51, 52, 53, 54, 58}
WM_LABELS = {2, 41, 77, 78, 79}
GM_LABELS = {
    3, 8, 9, 10, 11, 12, 13, 16, 17, 18, 26,
    42, 47, 48, 49, 50, 51, 52, 53, 54, 58,
}
CSF_LABELS = {4, 5, 14, 15, 24, 31, 43, 44, 63}


# DKT cortical labels are broad ranges. This lightweight rule is sufficient
# for building soft anatomy context maps in this implementation.
def is_cortex_label(x: np.ndarray) -> np.ndarray:
    return ((x >= 1000) & (x < 3000)).astype(np.float32)


def compute_boundary(binary_mask: np.ndarray) -> np.ndarray:
    binary_mask = binary_mask.astype(bool)
    if not binary_mask.any():
        return np.zeros_like(binary_mask, dtype=np.float32)
    eroded = ndi.binary_erosion(binary_mask, iterations=1, border_value=0)
    boundary = binary_mask ^ eroded
    return boundary.astype(np.float32)


def normalized_distance(mask: np.ndarray) -> np.ndarray:
    mask = mask.astype(bool)
    if not mask.any():
        return np.zeros_like(mask, dtype=np.float32)
    dist = ndi.distance_transform_edt(~mask).astype(np.float32)
    maxv = float(dist.max())
    if maxv > 0:
        dist /= maxv
    return dist


def build_coarse_region_map(dkt: np.ndarray, aseg: np.ndarray, ventricle: np.ndarray, cortex: np.ndarray) -> np.ndarray:
    region = np.zeros_like(aseg, dtype=np.int64)

    deep_gray = np.isin(aseg, list(DEEP_GRAY_LABELS))
    wm = np.isin(aseg, list(WM_LABELS))

    left_mask = ((aseg >= 1) & (aseg < 40)) | ((dkt >= 1000) & (dkt < 2000))
    right_mask = ((aseg >= 40) & (aseg < 80)) | ((dkt >= 2000) & (dkt < 3000))
    periventricular = ndi.binary_dilation(ventricle.astype(bool), iterations=3) & (~ventricle.astype(bool))

    region[left_mask & wm] = 1
    region[right_mask & wm] = 2
    region[cortex.astype(bool)] = 3
    region[periventricular] = 4
    region[deep_gray] = 5
    return region


def derive_anatomy_priors_from_fastsurfer(case: Dict, verbose: bool = False):
    """
    Build priors in FastSurfer space, then resample them into BRATS T1 space.
    """
    priors_fs, region_fs, fs_affine, fs_header = derive_anatomy_priors_in_fastsurfer_space(case)

    # BRATS T1 is the reference space.
    ref_img = load_nib(case['mri']['t1'])

    priors_brats, region_brats = resample_priors_and_region_to_brats_space(
        priors=priors_fs,
        region=region_fs,
        src_affine=fs_affine,
        ref_img=ref_img,
        src_header=fs_header,
    )

    # Normalize distance-map channels after resampling into BRATS space.
    # Channel 6 = wm_dist
    # Channel 7 = vent_dist
    priors_brats[6] = normalize_distance_channel(priors_brats[6])
    priors_brats[7] = normalize_distance_channel(priors_brats[7])

    if verbose:
        ref_shape = np.asanyarray(ref_img.dataobj).shape
        print(f"[Resampling check] MRI shape: {ref_shape}")
        print(f"[Resampling check] resampled priors shape: {priors_brats.shape}")
        print(f"[Resampling check] resampled region shape: {region_brats.shape}")

    return priors_brats, region_brats


def get_cached_prior_paths(cfg: TrainConfig, case_id: str) -> Tuple[str, str]:
    case_dir = os.path.join(cfg.cache_dir, case_id)
    ensure_dir(case_dir)
    return os.path.join(case_dir, 'priors.npy'), os.path.join(case_dir, 'region.npy')


def load_or_build_priors(cfg: TrainConfig, case: Dict):
    case_cache_dir = os.path.join(cfg.cache_dir, case['id'])
    priors_cache = os.path.join(case_cache_dir, 'priors.npy')
    region_cache = os.path.join(case_cache_dir, 'region.npy')

    if cfg.cache_priors and os.path.exists(priors_cache) and os.path.exists(region_cache):
        try:
            priors = np.load(priors_cache).astype(np.float32, copy=False)
            region = np.load(region_cache).astype(np.int32, copy=False)
            return priors, region
        except Exception as e:
            print(
                f"[Warning] Cache load failed for case {case['id']}. "
                f"Recomputing priors and rewriting cache. Error: {repr(e)}"
            )

    priors, region = derive_anatomy_priors_from_fastsurfer(case, verbose=False)

    if cfg.cache_priors:
        os.makedirs(case_cache_dir, exist_ok=True)
        np.save(priors_cache, priors)
        np.save(region_cache, region)

    return priors, region

def precompute_all_priors(cfg: TrainConfig, cases: List[Dict]) -> None:
    ensure_dir(cfg.cache_dir)
    for idx, case in enumerate(cases):
        load_or_build_priors(cfg, case)
        if (idx + 1) % 25 == 0 or (idx + 1) == len(cases):
            print(f'Cached priors for {idx + 1}/{len(cases)} cases')

In [ ]:
class BratsFastSurferDataset(Dataset):
    def __init__(
        self,
        cfg: TrainConfig,
        samples: Sequence[Dict],
        training: bool = True,
    ) -> None:
        self.cfg = cfg
        self.samples = list(samples)
        self.training = training

    def __len__(self) -> int:
        return len(self.samples)

    def _pad_if_needed(self, arr: np.ndarray, target: Tuple[int, int, int], value: float = 0) -> np.ndarray:
        dz = max(target[0] - arr.shape[-3], 0)
        dy = max(target[1] - arr.shape[-2], 0)
        dx = max(target[2] - arr.shape[-1], 0)
        if dz == 0 and dy == 0 and dx == 0:
            return arr
        pad_width = [(0, 0)] * arr.ndim
        pad_width[-3] = (dz // 2, dz - dz // 2)
        pad_width[-2] = (dy // 2, dy - dy // 2)
        pad_width[-1] = (dx // 2, dx - dx // 2)
        return np.pad(arr, pad_width, mode='constant', constant_values=value)

    def _center_crop(self, arr: np.ndarray, target: Tuple[int, int, int]) -> np.ndarray:
        z, y, x = arr.shape[-3:]
        tz, ty, tx = target
        sz = max((z - tz) // 2, 0)
        sy = max((y - ty) // 2, 0)
        sx = max((x - tx) // 2, 0)
        return arr[..., sz:sz + tz, sy:sy + ty, sx:sx + tx]

    def _random_crop(self, arr: np.ndarray, start: Tuple[int, int, int], target: Tuple[int, int, int]) -> np.ndarray:
        sz, sy, sx = start
        tz, ty, tx = target
        return arr[..., sz:sz + tz, sy:sy + ty, sx:sx + tx]

    def _sample_start(self, shape: Tuple[int, int, int], target: Tuple[int, int, int]) -> Tuple[int, int, int]:
        starts = []
        for dim, tgt in zip(shape, target):
            if dim <= tgt:
                starts.append(0)
            else:
                starts.append(random.randint(0, dim - tgt))
        return tuple(starts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        sample = self.samples[idx]

        t1 = zscore_nonzero(load_volume(sample['mri']['t1'], dtype=np.float32))
        t1ce = zscore_nonzero(load_volume(sample['mri']['t1ce'], dtype=np.float32))
        t2 = zscore_nonzero(load_volume(sample['mri']['t2'], dtype=np.float32))
        flair = zscore_nonzero(load_volume(sample['mri']['flair'], dtype=np.float32))
        seg = remap_brats_labels(load_volume(sample['seg'], dtype=np.int32)).astype(np.int64)

        x = np.stack([t1, t1ce, t2, flair], axis=0).astype(np.float32, copy=False)

        # Priors + region should already be derived and resampled into BRATS space
        priors, region = load_or_build_priors(self.cfg, sample)

        if idx == 0 and getattr(self.cfg, 'debug_resampling_shapes', False):
            print(f"[Dataset check] MRI shape: {x.shape[1:]}")
            print(f"[Dataset check] seg shape: {seg.shape}")
            print(f"[Dataset check] resampled priors shape: {priors.shape}")
            print(f"[Dataset check] resampled region shape: {region.shape}")

        x = self._pad_if_needed(x, self.cfg.patch_size, value=0)
        priors = self._pad_if_needed(priors, self.cfg.patch_size, value=0)
        seg = self._pad_if_needed(seg, self.cfg.patch_size, value=0)
        region = self._pad_if_needed(region, self.cfg.patch_size, value=0)

        if self.cfg.use_patch_sampling and self.training:
            start = self._sample_start(seg.shape[-3:], self.cfg.patch_size)
            x = self._random_crop(x, start, self.cfg.patch_size)
            priors = self._random_crop(priors, start, self.cfg.patch_size)
            seg = self._random_crop(seg, start, self.cfg.patch_size)
            region = self._random_crop(region, start, self.cfg.patch_size)
        else:
            x = self._center_crop(x, self.cfg.patch_size)
            priors = self._center_crop(priors, self.cfg.patch_size)
            seg = self._center_crop(seg, self.cfg.patch_size)
            region = self._center_crop(region, self.cfg.patch_size)

        return {
            'id': sample['id'],
            'image': torch.from_numpy(x).float(),
            'priors': torch.from_numpy(priors).float(),
            'region': torch.from_numpy(region).long(),
            'seg': torch.from_numpy(seg).long(),
        }

In [ ]:
class ConvNormAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ResidualBlock3D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = ConvNormAct(in_ch, out_ch, k=3, s=stride, p=1)
        self.conv2 = nn.Sequential(
            nn.Conv3d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch),
        )
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.InstanceNorm3d(out_ch),
            )
        else:
            self.skip = nn.Identity()
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.skip(x)
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.act(out + identity)
        return out


class MRIEncoder3D(nn.Module):
    def __init__(self, in_channels: int = 4, base: int = 32):
        super().__init__()
        self.e0 = nn.Sequential(ConvNormAct(in_channels, base), ResidualBlock3D(base, base))
        self.e1 = ResidualBlock3D(base, base * 2, stride=2)
        self.e2 = ResidualBlock3D(base * 2, base * 4, stride=2)
        self.e3 = ResidualBlock3D(base * 4, base * 8, stride=2)
        self.e4 = ResidualBlock3D(base * 8, 320, stride=2)

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        f0 = self.e0(x)
        f1 = self.e1(f0)
        f2 = self.e2(f1)
        f3 = self.e3(f2)
        f4 = self.e4(f3)
        return [f0, f1, f2, f3, f4]


class AnatomyEncoder3D(nn.Module):
    def __init__(self, in_channels: int = 9, base: int = 16):
        super().__init__()
        self.a0 = nn.Sequential(ConvNormAct(in_channels, base), ResidualBlock3D(base, base))
        self.a1 = ResidualBlock3D(base, base * 2, stride=2)
        self.a2 = ResidualBlock3D(base * 2, base * 4, stride=2)
        self.a3 = ResidualBlock3D(base * 4, base * 8, stride=2)
        self.a4 = ResidualBlock3D(base * 8, 160, stride=2)

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        g0 = self.a0(x)
        g1 = self.a1(g0)
        g2 = self.a2(g1)
        g3 = self.a3(g2)
        g4 = self.a4(g3)
        return [g0, g1, g2, g3, g4]


class AnatomyModulationBlock(nn.Module):
    def __init__(self, feat_ch: int, anat_ch: int):
        super().__init__()
        self.anat_proj = nn.Conv3d(anat_ch, feat_ch, kernel_size=1)
        self.spatial = nn.Sequential(
            nn.Conv3d(feat_ch * 2, feat_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(feat_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(feat_ch, feat_ch, kernel_size=1),
            nn.Sigmoid(),
        )
        hidden = max(feat_ch // 2, 16)
        self.film = nn.Sequential(
            nn.Linear(anat_ch, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, feat_ch * 2),
        )
        self.refine = ResidualBlock3D(feat_ch, feat_ch)

    def forward(self, x: torch.Tensor, anat: torch.Tensor) -> torch.Tensor:
        anat_p = self.anat_proj(anat)
        attn = self.spatial(torch.cat([x, anat_p], dim=1))
        out = x * (1.0 + attn)
        z = F.adaptive_avg_pool3d(anat, 1).flatten(1)
        gamma_beta = self.film(z)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        beta = beta.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        out = gamma * out + beta
        out = self.refine(out)
        return out


class DecoderBlock3D(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, anat_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2)
        self.merge = ResidualBlock3D(out_ch + skip_ch, out_ch)
        self.mod = AnatomyModulationBlock(out_ch, anat_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor, anat: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[-3:] != skip.shape[-3:]:
            x = F.interpolate(x, size=skip.shape[-3:], mode='trilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.merge(x)
        x = self.mod(x, anat)
        return x


class SharedTrunkHead(nn.Module):
    def __init__(self, in_ch: int = 32, out_ch: int = 64):
        super().__init__()
        self.block = nn.Sequential(ResidualBlock3D(in_ch, in_ch), ConvNormAct(in_ch, out_ch, k=3, s=1, p=1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ContextExpertHead(nn.Module):
    def __init__(self, trunk_ch: int = 64, context_ch: int = 128, num_classes: int = 4):
        super().__init__()
        self.compress = ConvNormAct(trunk_ch + context_ch, 64)
        self.refine = ResidualBlock3D(64, 64)
        self.out = nn.Conv3d(64, num_classes, kernel_size=1)

    def forward(self, trunk: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        if context.shape[-3:] != trunk.shape[-3:]:
            context = F.interpolate(context, size=trunk.shape[-3:], mode='trilinear', align_corners=False)
        x = torch.cat([trunk, context], dim=1)
        x = self.compress(x)
        x = self.refine(x)
        return self.out(x)


class BoundaryExpertHead(nn.Module):
    def __init__(self, trunk_ch: int = 64, boundary_ch: int = 16, edge_ch: int = 16, num_classes: int = 4):
        super().__init__()
        self.edge_proj = ConvNormAct(32, edge_ch)
        self.boundary_proj = ConvNormAct(boundary_ch, boundary_ch)
        self.refine1 = ResidualBlock3D(trunk_ch + boundary_ch + edge_ch, 64)
        self.refine2 = ResidualBlock3D(64, 64)
        self.out = nn.Conv3d(64, num_classes, kernel_size=1)

    def forward(self, trunk: torch.Tensor, early_feat: torch.Tensor, boundary_feat: torch.Tensor) -> torch.Tensor:
        edge = self.edge_proj(early_feat)
        if edge.shape[-3:] != trunk.shape[-3:]:
            edge = F.interpolate(edge, size=trunk.shape[-3:], mode='trilinear', align_corners=False)
        boundary = self.boundary_proj(boundary_feat)
        if boundary.shape[-3:] != trunk.shape[-3:]:
            boundary = F.interpolate(boundary, size=trunk.shape[-3:], mode='trilinear', align_corners=False)
        x = torch.cat([trunk, edge, boundary], dim=1)
        x = self.refine1(x)
        x = self.refine2(x)
        return self.out(x)


class SEBlock3D(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = F.adaptive_avg_pool3d(x, 1).flatten(1)
        z = F.relu(self.fc1(z), inplace=True)
        z = torch.sigmoid(self.fc2(z)).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x * z


class AnatomyAwareExpertHead(nn.Module):
    def __init__(self, trunk_ch: int = 64, anat_ch: int = 16, region_vocab: int = 16, region_dim: int = 8, num_classes: int = 4):
        super().__init__()
        self.region_embed = nn.Embedding(region_vocab, region_dim)
        self.anat_proj = ConvNormAct(anat_ch, anat_ch)
        self.refine1 = ResidualBlock3D(trunk_ch + anat_ch + region_dim, 64)
        self.se = SEBlock3D(64)
        self.refine2 = ResidualBlock3D(64, 64)
        self.out = nn.Conv3d(64, num_classes, kernel_size=1)

    def forward(self, trunk: torch.Tensor, anat_feat: torch.Tensor, region: torch.Tensor) -> torch.Tensor:
        anat = self.anat_proj(anat_feat)
        if anat.shape[-3:] != trunk.shape[-3:]:
            anat = F.interpolate(anat, size=trunk.shape[-3:], mode='trilinear', align_corners=False)
        region_emb = self.region_embed(region.clamp_min(0))
        region_emb = region_emb.permute(0, 4, 1, 2, 3).contiguous().float()
        if region_emb.shape[-3:] != trunk.shape[-3:]:
            region_emb = F.interpolate(region_emb, size=trunk.shape[-3:], mode='nearest')
        x = torch.cat([trunk, anat, region_emb], dim=1)
        x = self.refine1(x)
        x = self.se(x)
        x = self.refine2(x)
        return self.out(x)


class FusionMLP(nn.Module):
    def __init__(self, input_dim: int, num_experts: int = 3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_experts),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ASWAFNet(nn.Module):
    def __init__(self, num_classes: int = 4, num_prior_channels: int = 9, region_vocab: int = 16):
        super().__init__()
        self.num_classes = num_classes
        self.mri_encoder = MRIEncoder3D(in_channels=4, base=32)
        self.anat_encoder = AnatomyEncoder3D(in_channels=num_prior_channels, base=16)

        self.d3 = DecoderBlock3D(320, 256, 256, anat_ch=128)
        self.d2 = DecoderBlock3D(256, 128, 128, anat_ch=64)
        self.d1 = DecoderBlock3D(128, 64, 64, anat_ch=32)
        self.d0 = DecoderBlock3D(64, 32, 32, anat_ch=16)

        self.shared_trunk = SharedTrunkHead(in_ch=32, out_ch=64)
        self.boundary_feature_proj = nn.Conv3d(16, 16, kernel_size=1)
        self.anat_feature_proj = nn.Conv3d(16, 16, kernel_size=1)

        self.context_expert = ContextExpertHead(trunk_ch=64, context_ch=128, num_classes=num_classes)
        self.boundary_expert = BoundaryExpertHead(trunk_ch=64, boundary_ch=16, edge_ch=16, num_classes=num_classes)
        self.anatomy_expert = AnatomyAwareExpertHead(trunk_ch=64, anat_ch=16, region_vocab=region_vocab, region_dim=8, num_classes=num_classes)

        self.fusion_mlp = FusionMLP(input_dim=320 + 160 + 3, num_experts=3)
        self.log_tau = nn.Parameter(torch.tensor(0.0))
        self.register_buffer('calibration_scores', torch.zeros(3))

    def set_calibration_scores(self, scores: torch.Tensor) -> None:
        if scores.shape != (3,):
            raise ValueError(f'Expected calibration scores shape (3,), got {scores.shape}')
        self.calibration_scores.copy_(scores.detach().float())

    def _entropy_summary(self, logits: torch.Tensor) -> torch.Tensor:
        probs = torch.softmax(logits, dim=1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=1)
        return entropy.mean(dim=(1, 2, 3), keepdim=False).unsqueeze(1)

    def forward(self, image: torch.Tensor, priors: torch.Tensor, region: torch.Tensor) -> Dict[str, torch.Tensor]:
        f0, f1, f2, f3, f4 = self.mri_encoder(image)
        g0, g1, g2, g3, g4 = self.anat_encoder(priors)

        d3 = self.d3(f4, f3, g3)
        d2 = self.d2(d3, f2, g2)
        d1 = self.d1(d2, f1, g1)
        d0 = self.d0(d1, f0, g0)

        trunk = self.shared_trunk(d0)
        boundary_feat = self.boundary_feature_proj(g0)
        anatomy_feat = self.anat_feature_proj(g0)

        logits_1 = self.context_expert(trunk, f2)
        logits_2 = self.boundary_expert(trunk, f0, boundary_feat)
        logits_3 = self.anatomy_expert(trunk, anatomy_feat, region)

        z_img = F.adaptive_avg_pool3d(f4, 1).flatten(1)
        z_anat = F.adaptive_avg_pool3d(g4, 1).flatten(1)
        u1 = self._entropy_summary(logits_1)
        u2 = self._entropy_summary(logits_2)
        u3 = self._entropy_summary(logits_3)
        fusion_in = torch.cat([z_img, z_anat, u1, u2, u3], dim=1)

        residual_scores = self.fusion_mlp(fusion_in)
        scores = residual_scores + self.calibration_scores.unsqueeze(0)
        tau = torch.exp(self.log_tau).clamp_min(1e-3)
        weights = torch.softmax(scores / tau, dim=1)

        fused_logits = (
            weights[:, 0].view(-1, 1, 1, 1, 1) * logits_1
            + weights[:, 1].view(-1, 1, 1, 1, 1) * logits_2
            + weights[:, 2].view(-1, 1, 1, 1, 1) * logits_3
        )

        return {
            'logits_fused': fused_logits,
            'logits_expert_1': logits_1,
            'logits_expert_2': logits_2,
            'logits_expert_3': logits_3,
            'weights': weights,
        }

In [ ]:
class AnatomyAwareExpertHead(nn.Module):
    """Anatomy expert with safe region-id clamping."""
    def __init__(self, trunk_ch: int = 64, anat_ch: int = 16, region_vocab: int = 16, region_dim: int = 8, num_classes: int = 4):
        super().__init__()
        self.region_embed = nn.Embedding(region_vocab, region_dim)
        self.anat_proj = ConvNormAct(anat_ch, anat_ch)
        self.refine1 = ResidualBlock3D(trunk_ch + anat_ch + region_dim, 64)
        self.se = SEBlock3D(64)
        self.refine2 = ResidualBlock3D(64, 64)
        self.out = nn.Conv3d(64, num_classes, kernel_size=1)

    def forward(self, trunk: torch.Tensor, anat_feat: torch.Tensor, region: torch.Tensor) -> torch.Tensor:
        anat = self.anat_proj(anat_feat)
        if anat.shape[-3:] != trunk.shape[-3:]:
            anat = F.interpolate(anat, size=trunk.shape[-3:], mode='trilinear', align_corners=False)

        max_region_id = self.region_embed.num_embeddings - 1
        region_safe = region.long().clamp(0, max_region_id)
        region_emb = self.region_embed(region_safe)  # [B, D, H, W, region_dim]
        region_emb = region_emb.permute(0, 4, 1, 2, 3).contiguous().float()
        if region_emb.shape[-3:] != trunk.shape[-3:]:
            region_emb = F.interpolate(region_emb, size=trunk.shape[-3:], mode='nearest')

        x = torch.cat([trunk, anat, region_emb], dim=1)
        x = self.refine1(x)
        x = self.se(x)
        x = self.refine2(x)
        return self.out(x)


class RegionReliabilityRouter(nn.Module):
    """
    Efficient region-level anatomy-conditioned calibrated expert router.

    Inputs
    ------
    logits_1/logits_2/logits_3: [B, C, D, H, W]
    region:                    [B, D, H, W], integer region ids
    priors:                    [B, 9, D, H, W]
    z_img:                     [B, 320]
    z_anat:                    [B, 160]

    Outputs
    -------
    region_weights:  [B, R, 3]
    router_scores:   [B, R, 3]
    voxel_weights:   [B, 3, D, H, W]
    region_fraction: [B, R]
    """
    def __init__(
        self,
        num_regions: int = 6,
        num_experts: int = 3,
        num_classes: int = 4,
        z_img_dim: int = 320,
        z_anat_dim: int = 160,
        region_dim: int = 8,
        global_dim: int = 16,
        hidden_dim: int = 32,
        lambda_calib: float = 1.0,
    ):
        super().__init__()
        self.num_regions = int(num_regions)
        self.num_experts = int(num_experts)
        self.num_classes = int(num_classes)
        self.lambda_calib = float(lambda_calib)

        self.region_embed = nn.Embedding(self.num_regions, region_dim)
        self.global_proj = nn.Sequential(
            nn.Linear(z_img_dim + z_anat_dim, global_dim),
            nn.ReLU(inplace=True),
        )

        # Per-region features:
        # entropy per expert        -> 3
        # confidence per expert     -> 3
        # region fraction           -> 1
        # anatomy-boundary fraction -> 1
        # region embedding          -> region_dim
        # compressed global context -> global_dim
        feature_dim = (2 * self.num_experts) + 2 + region_dim + global_dim

        self.score_mlp = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, self.num_experts),
        )

        # Start near uniform routing while allowing immediate gradient flow through router features.
        nn.init.normal_(self.score_mlp[-1].weight, mean=0.0, std=1e-3)
        nn.init.zeros_(self.score_mlp[-1].bias)

        self.log_tau = nn.Parameter(torch.tensor(0.0))
        self.register_buffer('calibration_scores', torch.zeros(self.num_experts, self.num_regions))

    def set_calibration_scores(self, scores: torch.Tensor) -> None:
        """
        Set calibration table C[e, r].

        Accepted shapes:
        - [3, R] preferred
        - [R, 3] transposed automatically
        - [3] old global calibration scores, broadcast to all regions
        """
        scores = scores.detach().float().to(self.calibration_scores.device)

        if scores.shape == (self.num_experts,):
            scores = scores.view(self.num_experts, 1).expand(self.num_experts, self.num_regions)
        elif scores.shape == (self.num_regions, self.num_experts):
            scores = scores.transpose(0, 1).contiguous()
        elif scores.shape != (self.num_experts, self.num_regions):
            raise ValueError(
                f'Expected calibration scores shape ({self.num_experts}, {self.num_regions}), '
                f'({self.num_regions}, {self.num_experts}), or ({self.num_experts},), got {tuple(scores.shape)}'
            )

        self.calibration_scores.copy_(scores)

    def get_calibration_scores(self) -> torch.Tensor:
        return self.calibration_scores.detach().clone()

    def _region_reduce_mean(
        self,
        values: torch.Tensor,
        region_flat: torch.Tensor,
        counts_safe: torch.Tensor,
    ) -> torch.Tensor:
        """
        values:      [B, E, N]
        region_flat: [B, N]
        counts_safe: [B, R]
        returns:     [B, E, R]
        """
        B, E, N = values.shape
        sums = values.new_zeros(B, E, self.num_regions)
        index = region_flat.unsqueeze(1).expand(B, E, N)
        sums.scatter_add_(dim=2, index=index, src=values)
        return sums / counts_safe.unsqueeze(1)

    def forward(
        self,
        logits_1: torch.Tensor,
        logits_2: torch.Tensor,
        logits_3: torch.Tensor,
        region: torch.Tensor,
        priors: torch.Tensor,
        z_img: torch.Tensor,
        z_anat: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        B, C, D, H, W = logits_1.shape
        E = self.num_experts
        R = self.num_regions
        N = D * H * W

        region = region.long().clamp(0, R - 1)
        region_flat = region.reshape(B, N)

        ones = torch.ones(B, N, device=region.device, dtype=torch.float32)
        counts = torch.zeros(B, R, device=region.device, dtype=torch.float32)
        counts.scatter_add_(dim=1, index=region_flat, src=ones)
        counts_safe = counts.clamp_min(1.0)
        region_fraction = counts / float(max(N, 1))  # [B, R]

        # Detach statistics so routing learns from expert reliability cues without adding large
        # uncertainty-statistic backprop graphs. Expert logits still get gradients through fusion.
        stats_logits_stack = torch.stack([logits_1, logits_2, logits_3], dim=1).detach().float()  # [B, 3, C, D, H, W]
        probs = torch.softmax(stats_logits_stack, dim=2)

        confidence = probs.max(dim=2).values  # [B, 3, D, H, W]
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=2)  # [B, 3, D, H, W]
        entropy = entropy / math.log(max(C, 2))

        entropy_flat = entropy.reshape(B, E, N)
        confidence_flat = confidence.reshape(B, E, N)

        entropy_region = self._region_reduce_mean(entropy_flat, region_flat, counts_safe)        # [B, 3, R]
        confidence_region = self._region_reduce_mean(confidence_flat, region_flat, counts_safe)  # [B, 3, R]

        if priors is not None and priors.dim() == 5 and priors.size(1) > 8:
            boundary = priors[:, 8].float()
            if boundary.shape[-3:] != (D, H, W):
                boundary = F.interpolate(
                    boundary.unsqueeze(1),
                    size=(D, H, W),
                    mode='trilinear',
                    align_corners=False,
                ).squeeze(1)
        else:
            boundary = torch.zeros(B, D, H, W, device=region.device, dtype=torch.float32)

        boundary_flat = boundary.reshape(B, N)
        boundary_sum = torch.zeros(B, R, device=region.device, dtype=torch.float32)
        boundary_sum.scatter_add_(dim=1, index=region_flat, src=boundary_flat)
        boundary_fraction = boundary_sum / counts_safe  # [B, R]

        entropy_region = entropy_region.permute(0, 2, 1).contiguous()        # [B, R, 3]
        confidence_region = confidence_region.permute(0, 2, 1).contiguous()  # [B, R, 3]
        region_fraction_feat = region_fraction.unsqueeze(-1)                 # [B, R, 1]
        boundary_fraction_feat = boundary_fraction.unsqueeze(-1)             # [B, R, 1]

        region_ids = torch.arange(R, device=region.device, dtype=torch.long)
        region_emb = self.region_embed(region_ids).unsqueeze(0).expand(B, R, -1)  # [B, R, region_dim]

        global_context = self.global_proj(torch.cat([z_img.float(), z_anat.float()], dim=1))  # [B, global_dim]
        global_context = global_context.unsqueeze(1).expand(B, R, -1)                         # [B, R, global_dim]

        router_features = torch.cat(
            [
                entropy_region,
                confidence_region,
                region_fraction_feat,
                boundary_fraction_feat,
                region_emb,
                global_context,
            ],
            dim=-1,
        )

        learned_scores = self.score_mlp(router_features)  # [B, R, 3]
        calib = self.calibration_scores.transpose(0, 1).unsqueeze(0)  # [1, R, 3]
        router_scores = learned_scores + self.lambda_calib * calib

        tau = torch.exp(self.log_tau).clamp_min(1e-3)
        region_weights = torch.softmax(router_scores / tau, dim=-1)  # [B, R, 3]

        weights_by_expert = region_weights.permute(0, 2, 1).contiguous()  # [B, 3, R]
        gather_index = region_flat.unsqueeze(1).expand(B, E, N)           # [B, 3, N]
        voxel_weights = torch.gather(weights_by_expert, dim=2, index=gather_index)
        voxel_weights = voxel_weights.reshape(B, E, D, H, W)              # [B, 3, D, H, W]

        return region_weights, router_scores, voxel_weights, region_fraction


class ACRSWAFNet(nn.Module):
    """
    acf_net-base:
    reference model backbone/experts + efficient anatomy-conditioned calibrated region routing.
    """
    def __init__(
        self,
        num_classes: int = 4,
        num_prior_channels: int = 9,
        num_regions: int = 6,
        region_vocab: int = 16,
        router_hidden: int = 32,
        router_region_dim: int = 8,
        router_global_dim: int = 16,
        router_calibration_lambda: float = 1.0,
        debug_shapes: bool = False,
        return_voxel_weights: bool = False,
    ):
        super().__init__()
        self.model_name = 'acf_net-base'
        self.num_classes = int(num_classes)
        self.num_regions = int(num_regions)
        self.debug_shapes = bool(debug_shapes)
        self.return_voxel_weights = bool(return_voxel_weights)
        self._voxel_weights_returned_once = False

        self.mri_encoder = MRIEncoder3D(in_channels=4, base=32)
        self.anat_encoder = AnatomyEncoder3D(in_channels=num_prior_channels, base=16)

        self.d3 = DecoderBlock3D(320, 256, 256, anat_ch=128)
        self.d2 = DecoderBlock3D(256, 128, 128, anat_ch=64)
        self.d1 = DecoderBlock3D(128, 64, 64, anat_ch=32)
        self.d0 = DecoderBlock3D(64, 32, 32, anat_ch=16)

        self.shared_trunk = SharedTrunkHead(in_ch=32, out_ch=64)
        self.boundary_feature_proj = nn.Conv3d(16, 16, kernel_size=1)
        self.anat_feature_proj = nn.Conv3d(16, 16, kernel_size=1)

        safe_region_vocab = max(int(region_vocab), self.num_regions)

        self.context_expert = ContextExpertHead(
            trunk_ch=64,
            context_ch=128,
            num_classes=num_classes,
        )
        self.boundary_expert = BoundaryExpertHead(
            trunk_ch=64,
            boundary_ch=16,
            edge_ch=16,
            num_classes=num_classes,
        )
        self.anatomy_expert = AnatomyAwareExpertHead(
            trunk_ch=64,
            anat_ch=16,
            region_vocab=safe_region_vocab,
            region_dim=8,
            num_classes=num_classes,
        )

        self.router = RegionReliabilityRouter(
            num_regions=self.num_regions,
            num_experts=3,
            num_classes=num_classes,
            z_img_dim=320,
            z_anat_dim=160,
            region_dim=router_region_dim,
            global_dim=router_global_dim,
            hidden_dim=router_hidden,
            lambda_calib=router_calibration_lambda,
        )

    def set_calibration_scores(self, scores: torch.Tensor) -> None:
        self.router.set_calibration_scores(scores)

    def get_calibration_scores(self) -> torch.Tensor:
        return self.router.get_calibration_scores()

    def _prepare_region(self, region: torch.Tensor, target_shape: Tuple[int, int, int]) -> torch.Tensor:
        if region.dim() == 5:
            if region.size(1) == 1:
                region = region[:, 0]
            else:
                region = torch.argmax(region, dim=1)

        if region.shape[-3:] != target_shape:
            region = F.interpolate(
                region.unsqueeze(1).float(),
                size=target_shape,
                mode='nearest',
            ).squeeze(1)

        return region.long().clamp(0, self.num_regions - 1)

    def forward(self, image: torch.Tensor, priors: torch.Tensor, region: torch.Tensor) -> Dict[str, torch.Tensor]:
        region = self._prepare_region(region, image.shape[-3:])

        f0, f1, f2, f3, f4 = self.mri_encoder(image)
        g0, g1, g2, g3, g4 = self.anat_encoder(priors)

        d3 = self.d3(f4, f3, g3)
        d2 = self.d2(d3, f2, g2)
        d1 = self.d1(d2, f1, g1)
        d0 = self.d0(d1, f0, g0)

        trunk = self.shared_trunk(d0)
        boundary_feat = self.boundary_feature_proj(g0)
        anatomy_feat = self.anat_feature_proj(g0)

        logits_1 = self.context_expert(trunk, f2)
        logits_2 = self.boundary_expert(trunk, f0, boundary_feat)
        logits_3 = self.anatomy_expert(trunk, anatomy_feat, region)

        z_img = F.adaptive_avg_pool3d(f4, 1).flatten(1)   # [B, 320]
        z_anat = F.adaptive_avg_pool3d(g4, 1).flatten(1)  # [B, 160]

        region_weights, router_scores, voxel_weights, region_fraction = self.router(
            logits_1=logits_1,
            logits_2=logits_2,
            logits_3=logits_3,
            region=region,
            priors=priors,
            z_img=z_img,
            z_anat=z_anat,
        )

        logits_stack = torch.stack([logits_1, logits_2, logits_3], dim=1)  # [B, 3, C, D, H, W]
        fused_logits = (voxel_weights.unsqueeze(2) * logits_stack).sum(dim=1)  # [B, C, D, H, W]

        # Preserve existing key "weights" as an image-level summary [B, 3].
        image_weights = (region_weights * region_fraction.unsqueeze(-1)).sum(dim=1)

        outputs = {
            'logits_fused': fused_logits,
            'logits_expert_1': logits_1,
            'logits_expert_2': logits_2,
            'logits_expert_3': logits_3,
            'weights': image_weights,
            'region_weights': region_weights,
            'router_scores': router_scores,
        }

        include_voxel_weights = self.return_voxel_weights or (
            self.debug_shapes and not self._voxel_weights_returned_once
        )
        if include_voxel_weights:
            outputs['voxel_weights'] = voxel_weights.detach()
            self._voxel_weights_returned_once = True

        return outputs


In [ ]:
class SoftDiceLoss(nn.Module):
    def __init__(self, num_classes: int, smooth: float = 1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        probs = torch.softmax(logits, dim=1)
        target_oh = F.one_hot(target, num_classes=self.num_classes).permute(0, 4, 1, 2, 3).float()
        dims = (0, 2, 3, 4)
        intersection = (probs * target_oh).sum(dims)
        denom = probs.sum(dims) + target_oh.sum(dims)
        dice = (2.0 * intersection + self.smooth) / (denom + self.smooth)
        return 1.0 - dice[1:].mean()


def dice_per_class_from_logits(logits: torch.Tensor, target: torch.Tensor, num_classes: int) -> Dict[str, float]:
    pred = torch.argmax(logits, dim=1)
    metrics = {}
    for c in range(1, num_classes):
        pred_c = (pred == c).float()
        target_c = (target == c).float()
        intersection = (pred_c * target_c).sum().item()
        denom = pred_c.sum().item() + target_c.sum().item()
        dice = (2.0 * intersection + 1e-5) / (denom + 1e-5)
        metrics[f'dice_class_{c}'] = float(dice)
    fg_values = [metrics[f'dice_class_{c}'] for c in range(1, num_classes)]
    metrics['mean_dice'] = float(sum(fg_values) / len(fg_values))
    return metrics


def compute_total_loss(outputs: Dict[str, torch.Tensor], target: torch.Tensor, dice_loss: nn.Module) -> Tuple[torch.Tensor, Dict[str, float]]:
    ce = nn.CrossEntropyLoss()
    fused = outputs['logits_fused']
    e1 = outputs['logits_expert_1']
    e2 = outputs['logits_expert_2']
    e3 = outputs['logits_expert_3']

    loss_fused = dice_loss(fused, target) + ce(fused, target)
    loss_e1 = dice_loss(e1, target) + ce(e1, target)
    loss_e2 = dice_loss(e2, target) + ce(e2, target)
    loss_e3 = dice_loss(e3, target) + ce(e3, target)
    total = loss_fused + 0.3 * (loss_e1 + loss_e2 + loss_e3)

    parts = {
        'loss_total': float(total.detach().item()),
        'loss_fused': float(loss_fused.detach().item()),
        'loss_e1': float(loss_e1.detach().item()),
        'loss_e2': float(loss_e2.detach().item()),
        'loss_e3': float(loss_e3.detach().item()),
    }
    return total, parts

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int, mode: str = 'max', min_delta: float = 1e-4):
        if mode not in {'max', 'min'}:
            raise ValueError("mode must be 'max' or 'min'")
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best_score = None
        self.num_bad_epochs = 0
        self.should_stop = False

    def step(self, current: float) -> bool:
        if self.best_score is None:
            self.best_score = current
            return False

        improved = (
            current > self.best_score + self.min_delta
            if self.mode == 'max'
            else current < self.best_score - self.min_delta
        )

        if improved:
            self.best_score = current
            self.num_bad_epochs = 0
        else:
            self.num_bad_epochs += 1
            if self.num_bad_epochs >= self.patience:
                self.should_stop = True

        return self.should_stop

    def state_dict(self) -> Dict:
        return {
            'patience': self.patience,
            'mode': self.mode,
            'min_delta': self.min_delta,
            'best_score': self.best_score,
            'num_bad_epochs': self.num_bad_epochs,
            'should_stop': self.should_stop,
        }

    def load_state_dict(self, state: Dict) -> None:
        self.patience = state.get('patience', self.patience)
        self.mode = state.get('mode', self.mode)
        self.min_delta = state.get('min_delta', self.min_delta)
        self.best_score = state.get('best_score', self.best_score)
        self.num_bad_epochs = state.get('num_bad_epochs', self.num_bad_epochs)
        self.should_stop = state.get('should_stop', self.should_stop)


def capture_rng_state() -> Dict[str, Any]:
    state = {
        'torch_rng_state': torch.get_rng_state(),
        'numpy_rng_state': np.random.get_state(),
        'python_random_state': random.getstate(),
    }
    state['cuda_rng_state_all'] = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    return state


def restore_rng_state(state: Dict[str, Any]) -> None:
    try:
        torch_state = state.get('torch_rng_state', None)
        if torch_state is not None:
            torch.set_rng_state(torch_state.cpu())

        np_state = state.get('numpy_rng_state', None)
        if np_state is not None:
            np.random.set_state(np_state)

        py_state = state.get('python_random_state', None)
        if py_state is not None:
            random.setstate(py_state)

        cuda_state = state.get('cuda_rng_state_all', None)
        if torch.cuda.is_available() and cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)
    except Exception as exc:
        print(f"[Checkpoint warning] RNG state could not be fully restored: {repr(exc)}")


def _safe_torch_load(path: str, map_location: str = 'cpu') -> Dict:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def inspect_state_dict_compatibility(model: nn.Module, loaded_state: Dict[str, torch.Tensor]) -> Dict[str, List]:
    current_state = model.state_dict()
    missing_keys = [k for k in current_state.keys() if k not in loaded_state]
    unexpected_keys = [k for k in loaded_state.keys() if k not in current_state]

    shape_mismatches = []
    for k in current_state.keys():
        if k in loaded_state:
            current_shape = tuple(current_state[k].shape)
            loaded_shape = tuple(loaded_state[k].shape)
            if current_shape != loaded_shape:
                shape_mismatches.append((k, loaded_shape, current_shape))

    return {
        'missing_keys': missing_keys,
        'unexpected_keys': unexpected_keys,
        'shape_mismatches': shape_mismatches,
    }


def format_mismatch_report(report: Dict[str, List], max_items: int = 20) -> str:
    lines = []
    missing = report.get('missing_keys', [])
    unexpected = report.get('unexpected_keys', [])
    mismatches = report.get('shape_mismatches', [])

    if missing:
        lines.append(f"Missing keys: {len(missing)}")
        for k in missing[:max_items]:
            lines.append(f"  missing: {k}")
        if len(missing) > max_items:
            lines.append(f"  ... {len(missing) - max_items} more missing keys")

    if unexpected:
        lines.append(f"Unexpected keys: {len(unexpected)}")
        for k in unexpected[:max_items]:
            lines.append(f"  unexpected: {k}")
        if len(unexpected) > max_items:
            lines.append(f"  ... {len(unexpected) - max_items} more unexpected keys")

    if mismatches:
        lines.append(f"Shape mismatches: {len(mismatches)}")
        for k, loaded_shape, current_shape in mismatches[:max_items]:
            lines.append(f"  shape: {k}: checkpoint {loaded_shape} vs model {current_shape}")
        if len(mismatches) > max_items:
            lines.append(f"  ... {len(mismatches) - max_items} more shape mismatches")

    return "\n".join(lines) if lines else "No state_dict incompatibilities detected."


def save_checkpoint(
    checkpoint_path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: Optional[torch.cuda.amp.GradScaler],
    epoch: int,
    best_metric: float,
    history: List[Dict],
    early_stopping: EarlyStopping,
    config: TrainConfig,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
) -> None:
    ensure_dir(str(Path(checkpoint_path).parent))
    state = {
        'epoch': int(epoch),
        'model_name': getattr(model, 'model_name', model.__class__.__name__),
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict() if optimizer is not None else None,
        'scheduler_state': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state': scaler.state_dict() if scaler is not None else None,
        'best_metric': float(best_metric),
        'history': history,
        'early_stopping': early_stopping.state_dict() if early_stopping is not None else None,
        'config': asdict(config),
        **capture_rng_state(),
    }
    torch.save(state, checkpoint_path)


def load_checkpoint(
    checkpoint_path: str,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scaler: Optional[torch.cuda.amp.GradScaler] = None,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
    map_location: str = 'cpu',
    strict: bool = True,
) -> Dict:
    ckpt = _safe_torch_load(checkpoint_path, map_location=map_location)
    if 'model_state' not in ckpt:
        raise RuntimeError(f"Checkpoint does not contain 'model_state': {checkpoint_path}")

    report = inspect_state_dict_compatibility(model, ckpt['model_state'])
    has_incompatibility = bool(report['missing_keys'] or report['unexpected_keys'] or report['shape_mismatches'])
    if strict and has_incompatibility:
        raise RuntimeError(
            "Checkpoint is incompatible with the current architecture.\n"
            + format_mismatch_report(report)
        )

    if report['shape_mismatches']:
        # PyTorch cannot load mismatched tensor shapes even with strict=False.
        current_state = model.state_dict()
        filtered_state = {
            k: v for k, v in ckpt['model_state'].items()
            if k in current_state and tuple(v.shape) == tuple(current_state[k].shape)
        }
        model.load_state_dict(filtered_state, strict=False)
    else:
        model.load_state_dict(ckpt['model_state'], strict=strict)

    if optimizer is not None and ckpt.get('optimizer_state') is not None:
        try:
            optimizer.load_state_dict(ckpt['optimizer_state'])
        except Exception as exc:
            print(f"[Checkpoint warning] Optimizer state could not be loaded: {repr(exc)}")

    if scheduler is not None and ckpt.get('scheduler_state') is not None:
        try:
            scheduler.load_state_dict(ckpt['scheduler_state'])
        except Exception as exc:
            print(f"[Checkpoint warning] Scheduler state could not be loaded: {repr(exc)}")

    if scaler is not None and ckpt.get('scaler_state') is not None:
        try:
            scaler.load_state_dict(ckpt['scaler_state'])
        except Exception as exc:
            print(f"[Checkpoint warning] AMP scaler state could not be loaded: {repr(exc)}")

    restore_rng_state(ckpt)
    return ckpt


def latest_checkpoint_path(checkpoint_dir: str) -> Optional[str]:
    path = Path(checkpoint_dir)
    if not path.exists():
        return None

    latest = path / 'latest.pt'
    if latest.exists():
        return str(latest)

    ckpts = sorted(path.glob('epoch_*.pt'))
    if ckpts:
        return str(ckpts[-1])

    return None


def prune_old_epoch_checkpoints(checkpoint_dir: str, keep_last_k: int = 3) -> None:
    if keep_last_k is None or keep_last_k <= 0:
        return
    path = Path(checkpoint_dir)
    if not path.exists():
        return
    ckpts = sorted(path.glob('epoch_*.pt'))
    old = ckpts[:-keep_last_k]
    for p in old:
        try:
            p.unlink()
        except Exception as exc:
            print(f"[Checkpoint warning] Could not remove old checkpoint {p}: {repr(exc)}")


In [ ]:
@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    dice_loss: nn.Module,
    num_classes: int,
    epoch: Optional[int] = None,
    debug_shapes: bool = False,
) -> Dict[str, float]:
    model.eval()
    loss_meter = AverageMeter()
    dice_meter = AverageMeter()
    class_meters = {f'dice_class_{c}': AverageMeter() for c in range(1, num_classes)}

    desc = f"Val epoch {epoch}" if epoch is not None else "Validation"
    pbar = tqdm(loader, desc=desc, leave=False)

    for batch_idx, batch in enumerate(pbar):
        image = batch['image'].to(device, non_blocking=True)
        priors = batch['priors'].to(device, non_blocking=True)
        region = batch['region'].to(device, non_blocking=True)
        target = batch['seg'].to(device, non_blocking=True)

        outputs = model(image, priors, region)
        if debug_shapes and batch_idx == 0:
            print_debug_shapes_once(image, priors, region, outputs, prefix='[DEBUG_SHAPES validation first batch]')

        loss, _ = compute_total_loss(outputs, target, dice_loss)
        metrics = dice_per_class_from_logits(outputs['logits_fused'], target, num_classes)

        bsz = image.size(0)
        loss_meter.update(float(loss.item()), bsz)
        dice_meter.update(metrics['mean_dice'], bsz)
        for c in range(1, num_classes):
            class_meters[f'dice_class_{c}'].update(metrics[f'dice_class_{c}'], bsz)

        pbar.set_postfix({
            'loss': f'{loss_meter.avg:.4f}',
            'dice': f'{dice_meter.avg:.4f}',
        })

    result = {'val_loss': loss_meter.avg, 'val_mean_dice': dice_meter.avg}
    for c in range(1, num_classes):
        result[f'val_dice_class_{c}'] = class_meters[f'dice_class_{c}'].avg
    return result


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: Optional[torch.cuda.amp.GradScaler],
    device: str,
    dice_loss: nn.Module,
    num_classes: int,
    amp: bool = True,
    grad_clip_norm: float = 12.0,
    epoch: Optional[int] = None,
    debug_shapes: bool = False,
) -> Dict[str, float]:
    model.train()
    loss_meter = AverageMeter()
    dice_meter = AverageMeter()

    desc = f"Train epoch {epoch}" if epoch is not None else "Training"
    pbar = tqdm(loader, desc=desc, leave=False)

    for batch_idx, batch in enumerate(pbar):
        image = batch['image'].to(device, non_blocking=True)
        priors = batch['priors'].to(device, non_blocking=True)
        region = batch['region'].to(device, non_blocking=True)
        target = batch['seg'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp and device.startswith('cuda')):
            outputs = model(image, priors, region)
            loss, _ = compute_total_loss(outputs, target, dice_loss)

        if debug_shapes and batch_idx == 0:
            print_debug_shapes_once(image, priors, region, outputs, prefix='[DEBUG_SHAPES training first batch]')

        if scaler is not None and amp and device.startswith('cuda'):
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            optimizer.step()

        if batch_idx == 0:
            print_gpu_memory('after first training batch')

        with torch.no_grad():
            metrics = dice_per_class_from_logits(outputs['logits_fused'], target, num_classes)
            bsz = image.size(0)
            loss_meter.update(float(loss.item()), bsz)
            dice_meter.update(metrics['mean_dice'], bsz)

        pbar.set_postfix({
            'loss': f'{loss_meter.avg:.4f}',
            'dice': f'{dice_meter.avg:.4f}',
        })

    return {'train_loss': loss_meter.avg, 'train_mean_dice': dice_meter.avg}


In [ ]:
def initialize_zero_region_calibration(model: nn.Module, cfg: TrainConfig) -> torch.Tensor:
    """
    acf_net-base starts with a zero region calibration table C[e, r].
    No extra validation/calibration pass is performed at startup.
    """
    scores = torch.zeros(3, cfg.num_regions, dtype=torch.float32, device=cfg.device)
    if hasattr(model, 'set_calibration_scores'):
        model.set_calibration_scores(scores)
    print(f"[Calibration] Initialized zero region calibration table with shape {tuple(scores.shape)}")
    return scores


In [ ]:
def build_dataloaders(cfg: TrainConfig) -> Tuple[DataLoader, DataLoader, Optional[DataLoader], Dict[str, List[Dict]]]:
    print("[Data] Loading or creating ACR manifest/splits...")
    print(f"[Data] Manifest path: {cfg.manifest_path}")
    print(f"[Data] Split path:    {cfg.split_path}")

    cases = load_or_create_manifest(cfg)
    splits = load_or_create_splits(cfg, cases)

    print("[Data] Building datasets...")
    train_ds = BratsFastSurferDataset(cfg=cfg, samples=splits['train'], training=True)
    val_ds = BratsFastSurferDataset(cfg=cfg, samples=splits['val'], training=False)
    test_ds = BratsFastSurferDataset(cfg=cfg, samples=splits['test'], training=False) if len(splits['test']) > 0 else None

    print(f"[Data] train cases: {len(train_ds)}")
    print(f"[Data] val cases:   {len(val_ds)}")
    print(f"[Data] test cases:  {len(test_ds) if test_ds is not None else 0}")
    print(f"[Data] batch_size={cfg.batch_size}, num_workers={cfg.num_workers}, patch_size={cfg.patch_size}")
    print(f"[Data] ACR prior cache dir: {cfg.cache_dir}")

    print("[Data] Creating DataLoaders...")
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
        prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=cfg.num_workers > 0,
    )
    test_loader = None
    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=1,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            persistent_workers=cfg.num_workers > 0,
        )

    print("[Data] DataLoaders ready.")
    return train_loader, val_loader, test_loader, splits


In [ ]:
def verify_dataset_roots(cfg: TrainConfig) -> None:
    print('Checking dataset roots...')
    print(f'BRATS root: {cfg.brats_root}')
    print(f'FastSurfer root: {cfg.fastsurfer_root}')

    brats_exists = os.path.isdir(cfg.brats_root)
    fs_exists = os.path.isdir(cfg.fastsurfer_root)
    print(f'BRATS root exists: {brats_exists}')
    print(f'FastSurfer root exists: {fs_exists}')

    if not brats_exists or not fs_exists:
        raise FileNotFoundError(
            'One or both dataset roots do not exist. These defaults assume Colab Drive is mounted under FILL IN WITH DIRECTORY FOR GOOGLE DRIVE ROOT/.'
        )


def print_matched_case_summary(cfg: TrainConfig, cases: Optional[List[Dict]] = None) -> List[Dict]:
    if cases is None:
        cases = load_or_create_manifest(cfg)
    print(f'Matched cases found: {len(cases)}')
    print(f"First 3 matched case IDs: {[c['id'] for c in cases[:3]]}")
    return cases


def inspect_sample_case_shapes(case: Dict) -> Dict[str, np.ndarray]:
    print(f"Inspecting raw sample case: {case['id']}")
    vols = {
        't1': load_volume(case['mri']['t1'], dtype=np.float32),
        't1ce': load_volume(case['mri']['t1ce'], dtype=np.float32),
        't2': load_volume(case['mri']['t2'], dtype=np.float32),
        'flair': load_volume(case['mri']['flair'], dtype=np.float32),
        'seg': load_volume(case['seg'], dtype=np.int32),
        'dkt': load_volume(case['fastsurfer']['dkt'], dtype=np.int32),
        'aseg': load_volume(case['fastsurfer']['aseg'], dtype=np.int32),
        'mask': load_volume(case['fastsurfer']['mask'], dtype=np.float32),
    }
    for name, arr in vols.items():
        print(f'{name:>6} shape: {arr.shape}')
    return vols


def inspect_sample_priors(cfg: TrainConfig, case: Dict) -> Tuple[np.ndarray, np.ndarray]:
    print(f"Building anatomy priors for sample case: {case['id']}")
    ref_t1 = load_volume(case['mri']['t1'], dtype=np.float32)
    priors, region = derive_anatomy_priors_from_fastsurfer(case, verbose=False)

    print(f'MRI shape: {ref_t1.shape}')
    print(f'resampled priors shape: {priors.shape}')
    print(f'resampled region shape: {region.shape}')
    print(f'unique region values: {np.unique(region).tolist()}')
    print(f'wm_dist min/max: {float(priors[6].min()):.6f} / {float(priors[6].max()):.6f}')
    print(f'vent_dist min/max: {float(priors[7].min()):.6f} / {float(priors[7].max()):.6f}')
    return priors, region


def plot_sample_case_slices(case: Dict, priors: Optional[np.ndarray] = None, region: Optional[np.ndarray] = None) -> None:
    t1ce = load_volume(case['mri']['t1ce'], dtype=np.float32)
    seg = remap_brats_labels(load_volume(case['seg'], dtype=np.int32))
    if priors is None or region is None:
        priors, region = derive_anatomy_priors_from_fastsurfer(case)

    z = t1ce.shape[0] // 2
    wm = priors[0]
    vent = priors[3]
    cortex = priors[5]
    anat_boundary = priors[8]

    items = [
        ('t1ce', t1ce[z], 'gray'),
        ('seg', seg[z], 'viridis'),
        ('wm prior', wm[z], 'gray'),
        ('ventricle prior', vent[z], 'gray'),
        ('cortex prior', cortex[z], 'gray'),
        ('anat_boundary', anat_boundary[z], 'gray'),
        ('region map', region[z], 'tab20'),
    ]

    fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4))
    if len(items) == 1:
        axes = [axes]
    for ax, (title, img, cmap) in zip(axes, items):
        ax.imshow(np.rot90(img), cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def sanity_check_dataloader(cfg: TrainConfig, splits: Optional[Dict[str, List[Dict]]] = None) -> None:
    if splits is None:
        cases = load_or_create_manifest(cfg)
        splits = load_or_create_splits(cfg, cases)
    ds = BratsFastSurferDataset(cfg=cfg, samples=splits['train'], training=True)
    loader = DataLoader(ds, batch_size=min(cfg.batch_size, 1), shuffle=False, num_workers=0, pin_memory=False)
    batch = next(iter(loader))
    print('Loaded 1 batch from DataLoader')
    print(f"image tensor shape:  {tuple(batch['image'].shape)}")
    print(f"priors tensor shape: {tuple(batch['priors'].shape)}")
    print(f"region tensor shape: {tuple(batch['region'].shape)}")
    print(f"seg tensor shape:    {tuple(batch['seg'].shape)}")
    print(f"case ids: {batch['id']}")


def run_pretraining_verification(cfg: TrainConfig) -> Dict[str, List[Dict]]:
    verify_dataset_roots(cfg)
    cases = load_or_create_manifest(cfg)
    print_matched_case_summary(cfg, cases)
    if len(cases) == 0:
        raise RuntimeError('No matched cases found after scanning BRATS and FastSurfer roots.')

    sample_case = cases[0]
    inspect_sample_case_shapes(sample_case)
    priors, region = inspect_sample_priors(cfg, sample_case)
    plot_sample_case_slices(sample_case, priors=priors, region=region)

    splits = load_or_create_splits(cfg, cases)
    sanity_check_dataloader(cfg, splits)
    return splits

In [ ]:
def fit(cfg: TrainConfig) -> None:
    mount_drive_in_colab()

    print("=" * 80)
    print("acf_net-base training startup")
    print("=" * 80)
    print(f"Device: {cfg.device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU: not available, using CPU")
    print(f"Smoke test mode: {cfg.smoke_test}")
    if cfg.smoke_test:
        print(f"Smoke test epochs: {cfg.smoke_test_epochs}")

    print("\n[Isolation] This acf_net-base run uses separate output files from the original reference model notebook.")
    print(f"[Isolation] work_dir:       {cfg.work_dir}")
    print(f"[Isolation] cache_dir:      {cfg.cache_dir}")
    print(f"[Isolation] checkpoint_dir: {cfg.checkpoint_dir}")
    print(f"[Isolation] log_dir:        {cfg.log_dir}")

    set_seed(cfg.seed)

    ensure_dir(cfg.work_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.checkpoint_dir)
    ensure_dir(cfg.log_dir)
    save_json(asdict(cfg), os.path.join(cfg.log_dir, "train_config.json"))

    print("\n[Startup] Building dataloaders...")
    train_loader, val_loader, _, splits = build_dataloaders(cfg)
    print(f"[Startup] Split sizes: { {k: len(v) for k, v in splits.items()} }")

    if cfg.cache_priors and cfg.precompute_all_priors_before_training:
        print("\n[Startup] Precomputing cached priors for all splits...")
        all_cases = splits["train"] + splits["val"] + splits["test"]
        for case in tqdm(all_cases, desc='Precomputing acf_net-base priors'):
            load_or_build_priors(cfg, case)

    print("\n[Startup] Initializing acf_net-base model...")
    model = ACRSWAFNet(
        num_classes=cfg.num_classes,
        num_prior_channels=9,
        num_regions=cfg.num_regions,
        region_vocab=cfg.region_vocab,
        router_hidden=cfg.router_hidden,
        router_region_dim=cfg.router_region_dim,
        router_global_dim=cfg.router_global_dim,
        router_calibration_lambda=cfg.router_calibration_lambda,
        debug_shapes=cfg.DEBUG_SHAPES,
        return_voxel_weights=False,
    ).to(cfg.device)

    print_parameter_summary(
        model,
        model_name='acf_net-base',
        baseline_trainable_m=30.43,
        print_modules=True,
    )
    print_gpu_memory('after model creation')

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp and cfg.device.startswith("cuda"))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.max_epochs)
    dice_loss = SoftDiceLoss(num_classes=cfg.num_classes)
    early_stopping = EarlyStopping(
        patience=cfg.patience,
        mode=cfg.monitor_mode,
        min_delta=cfg.min_delta,
    )

    # V1: zero calibration table. No expensive validation/calibration startup pass.
    initialize_zero_region_calibration(model, cfg)

    start_epoch = 0
    best_metric = -math.inf if cfg.monitor_mode == "max" else math.inf
    history: List[Dict] = []

    resume_path = cfg.resume_path
    if resume_path is None and cfg.AUTO_RESUME:
        resume_path = latest_checkpoint_path(cfg.checkpoint_dir)

    if resume_path is not None and os.path.exists(resume_path):
        print(f"\n[Auto-resume] Candidate checkpoint found: {resume_path}")
        try:
            ckpt = load_checkpoint(
                resume_path,
                model,
                optimizer=optimizer,
                scaler=scaler,
                scheduler=scheduler,
                map_location=cfg.device,
                strict=True,
            )
            completed_epoch = int(ckpt["epoch"])
            start_epoch = completed_epoch + 1
            best_metric = float(ckpt.get("best_metric", best_metric))
            history = list(ckpt.get("history", []))
            if ckpt.get("early_stopping") is not None:
                early_stopping.load_state_dict(ckpt["early_stopping"])

            print("[Auto-resume] Resume successful.")
            print(f"[Auto-resume] checkpoint path:       {resume_path}")
            print(f"[Auto-resume] resumed epoch:        {completed_epoch}")
            print(f"[Auto-resume] completed epochs:     {len(history)}")
            print(f"[Auto-resume] best metric so far:   {best_metric:.6f}")
            print(f"[Auto-resume] next epoch to run:    {start_epoch}")
        except RuntimeError as exc:
            print("\n[Auto-resume] Checkpoint exists but is incompatible with the current ACR architecture.")
            print(str(exc))
            if cfg.resume_path is not None:
                raise
            print("[Auto-resume] Starting from scratch. The incompatible checkpoint was not modified.")
    else:
        print("\n[Auto-resume] No valid latest checkpoint found. Training is starting from scratch.")

    history_path = os.path.join(cfg.log_dir, "history.json")
    effective_max_epochs = cfg.smoke_test_epochs if cfg.smoke_test else cfg.max_epochs

    print("\n[Startup] Starting training loop...")

    for epoch in range(start_epoch, effective_max_epochs):
        epoch_start = time.time()

        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
            cfg.device,
            dice_loss,
            cfg.num_classes,
            amp=cfg.amp,
            grad_clip_norm=cfg.grad_clip_norm,
            epoch=epoch,
            debug_shapes=cfg.DEBUG_SHAPES and epoch == start_epoch,
        )

        val_metrics = validate_one_epoch(
            model,
            val_loader,
            cfg.device,
            dice_loss,
            cfg.num_classes,
            epoch=epoch,
            debug_shapes=False,
        )

        scheduler.step()

        epoch_time = time.time() - epoch_start
        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "time_sec": epoch_time,
            **train_metrics,
            **val_metrics,
        }
        history.append(row)
        save_json({"history": history}, history_path)

        current_metric = row[cfg.monitor_metric]
        improved = (
            current_metric > best_metric + cfg.min_delta
            if cfg.monitor_mode == "max"
            else current_metric < best_metric - cfg.min_delta
        )
        if improved:
            best_metric = float(current_metric)

        stop = early_stopping.step(float(current_metric))

        if improved:
            best_path = os.path.join(cfg.checkpoint_dir, "best_model.pt")
            save_checkpoint(
                best_path,
                model,
                optimizer,
                scaler,
                epoch,
                best_metric,
                history,
                early_stopping,
                cfg,
                scheduler=scheduler,
            )
            print(f"[Checkpoint] Saved best model: {best_path}")

        save_epoch_file = cfg.SAVE_EVERY_EPOCH or (cfg.save_every > 0 and ((epoch + 1) % cfg.save_every == 0))
        if save_epoch_file and not cfg.SAVE_BEST_ONLY:
            ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch:04d}.pt")
            save_checkpoint(
                ckpt_path,
                model,
                optimizer,
                scaler,
                epoch,
                best_metric,
                history,
                early_stopping,
                cfg,
                scheduler=scheduler,
            )
            prune_old_epoch_checkpoints(cfg.checkpoint_dir, keep_last_k=cfg.keep_last_k)

        latest_path = os.path.join(cfg.checkpoint_dir, "latest.pt")
        save_checkpoint(
            latest_path,
            model,
            optimizer,
            scaler,
            epoch,
            best_metric,
            history,
            early_stopping,
            cfg,
            scheduler=scheduler,
        )

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={row['train_loss']:.4f} | "
            f"train_dice={row['train_mean_dice']:.4f} | "
            f"val_loss={row['val_loss']:.4f} | "
            f"val_mean_dice={row['val_mean_dice']:.4f} | "
            f"best={best_metric:.4f} | "
            f"lr={row['lr']:.2e} | "
            f"time={epoch_time:.1f}s"
        )

        if stop:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    print("Training finished.")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
try:
    cfg
except NameError:
    cfg = TrainConfig()

model = ACRSWAFNet(
    num_classes=cfg.num_classes,
    num_prior_channels=9,
    num_regions=cfg.num_regions,
    region_vocab=cfg.region_vocab,
    router_hidden=cfg.router_hidden,
    router_region_dim=cfg.router_region_dim,
    router_global_dim=cfg.router_global_dim,
    router_calibration_lambda=cfg.router_calibration_lambda,
    debug_shapes=cfg.DEBUG_SHAPES,
).to(cfg.device)

print_parameter_summary(model, model_name='acf_net-base', baseline_trainable_m=30.43, print_modules=True)
print_gpu_memory('after optional acf_net-base sanity creation')


In [ ]:
try:
    cfg
except NameError:
    cfg = TrainConfig()

print("ACR work_dir:", cfg.work_dir)
print("ACR cache_dir:", cfg.cache_dir)
print("ACR checkpoint_dir:", cfg.checkpoint_dir)
print("ACR latest checkpoint candidate:", latest_checkpoint_path(cfg.checkpoint_dir))


In [ ]:
from google.colab import drive
drive.mount('FILL IN WITH DIRECTORY FOR COLAB DRIVE MOUNT', force_remount=True)

RUN TO TRAIN

In [ ]:
if __name__ == '__main__':
    cfg = TrainConfig(
        brats_root="FILL IN WITH DIRECTORY FOR BRATS GLI TRAINING DATA",
        fastsurfer_root="FILL IN WITH DIRECTORY FOR FASTSURFER LABELED TRAINING OUTPUTS",
        use_patch_sampling=False,
        patch_size=(96, 96, 96),
        batch_size=1,

        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,

        cache_priors=True,
        smoke_test=False,
        AUTO_RESUME=True,
        DEBUG_SHAPES=True,
        SAVE_EVERY_EPOCH=True,
        SAVE_BEST_ONLY=False,

        checkpoint_dir="FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS",
        cache_dir="FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE",
        log_dir="FILL IN WITH DIRECTORY FOR TRAINING LOGS",
        manifest_path="FILL IN WITH DIRECTORY FOR RUN MANIFEST/manifest.json",
        split_path="FILL IN WITH DIRECTORY FOR TRAIN VALIDATION TEST SPLITS/splits.json",
    )

    print("acf_net-base isolated paths:")
    print("  work_dir:      ", cfg.work_dir)
    print("  cache_dir:     ", cfg.cache_dir)
    print("  checkpoint_dir:", cfg.checkpoint_dir)
    print("  log_dir:       ", cfg.log_dir)

    run_pretraining_verification(cfg)
    fit(cfg)

RUN TO EVAL

In [ ]:
from pathlib import Path
import os
import json
import copy
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    from sklearn.metrics import roc_auc_score, average_precision_score
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    print(f"[Warning] sklearn metrics unavailable: {repr(exc)}")

try:
    from scipy.ndimage import binary_erosion, distance_transform_edt
    SCIPY_AVAILABLE = True
except Exception as exc:
    SCIPY_AVAILABLE = False
    print(f"[Warning] scipy HD95 helpers unavailable: {repr(exc)}")

eval_cfg = TrainConfig()

# ACR-only run paths
eval_cfg.work_dir = "FILL IN WITH DIRECTORY FOR ACF NET BASE BRATS RUNS"
eval_cfg.checkpoint_dir = "FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS"
eval_cfg.log_dir = "FILL IN WITH DIRECTORY FOR TRAINING LOGS"
eval_cfg.cache_dir = "FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE"
eval_cfg.manifest_path = "FILL IN WITH DIRECTORY FOR RUN MANIFEST/manifest.json"
eval_cfg.split_path = "FILL IN WITH DIRECTORY FOR TRAIN VALIDATION TEST SPLITS/splits.json"
eval_cfg.results_dir = "FILL IN WITH DIRECTORY FOR EVALUATION RESULTS"

# Best ACR checkpoint only
eval_cfg.best_checkpoint_path = (
    "FILL IN WITH DIRECTORY FOR BEST MODEL CHECKPOINT/best_model.pt"
)

# Labeled test evaluation should use FastSurfer outputs overlapping the labeled 1251-case split.
eval_cfg.fastsurfer_labeled_root = (
    "FILL IN WITH DIRECTORY FOR FASTSURFER LABELED TRAINING OUTPUTS"
)

# External inference-only FastSurfer root, not used for labeled metrics unless labels are later matched.
eval_cfg.fastsurfer_external_root = (
    "FILL IN WITH DIRECTORY FOR FASTSURFER TEST OUTPUTS"
)

# Isolated caches
eval_cfg.val_cache_dir = "FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE"
eval_cfg.test_cache_dir = "FILL IN WITH DIRECTORY FOR LABELED TEST PRIOR CACHE"
eval_cfg.external_cache_dir = "FILL IN WITH DIRECTORY FOR EXTERNAL PRIOR CACHE"

eval_cfg.batch_size = 1
eval_cfg.num_workers = 2
eval_cfg.use_patch_sampling = False
eval_cfg.DEBUG_SHAPES = False
eval_cfg.debug_resampling_shapes = False

ensure_dir(eval_cfg.results_dir)
ensure_dir(eval_cfg.test_cache_dir)
ensure_dir(eval_cfg.external_cache_dir)

print("=" * 80)
print("acf_net-base final labeled evaluation config")
print("=" * 80)
print(f"Best checkpoint:          {eval_cfg.best_checkpoint_path}")
print(f"Split file:               {eval_cfg.split_path}")
print(f"Validation prior cache:   {eval_cfg.val_cache_dir}")
print(f"Labeled test FS root:     {eval_cfg.fastsurfer_labeled_root}")
print(f"Labeled test prior cache: {eval_cfg.test_cache_dir}")
print(f"External FS root:         {eval_cfg.fastsurfer_external_root}")
print(f"Results dir:              {eval_cfg.results_dir}")

assert os.path.exists(eval_cfg.best_checkpoint_path), (
    f"Best checkpoint not found: {eval_cfg.best_checkpoint_path}"
)
assert os.path.exists(eval_cfg.split_path), (
    f"Split file not found: {eval_cfg.split_path}"
)
assert os.path.exists(eval_cfg.fastsurfer_labeled_root), (
    f"Labeled FastSurfer root not found: {eval_cfg.fastsurfer_labeled_root}"
)

In [ ]:
FASTSURFER_REQUIRED_FILES = {
    "dkt": "aparc.DKTatlas+aseg.deep.mgz",
    "aseg": "aseg.auto_noCCseg.mgz",
    "mask": "mask.mgz",
}


def _path_exists_and_valid(path_obj):
    path_obj = Path(path_obj)
    return path_obj.exists() and path_obj.is_file() and not str(path_obj).endswith(":Zone.Identifier")


def find_fastsurfer_case_dir(case_id, fastsurfer_root):
    """
    Supports both layouts:

    1. <root>/<case_id>/mri/aseg.auto_noCCseg.mgz
    2. <root>/<case_id>/aseg.auto_noCCseg.mgz

    Returns:
        case_mri_dir: Path or None
        layout_name: str
        missing_keys: list[str]
    """
    root = Path(fastsurfer_root)

    candidates = [
        (root / case_id / "mri", "case/mri"),
        (root / case_id, "case_root"),
    ]

    for candidate_dir, layout_name in candidates:
        files = {
            key: candidate_dir / filename
            for key, filename in FASTSURFER_REQUIRED_FILES.items()
        }
        missing = [key for key, path in files.items() if not _path_exists_and_valid(path)]
        if len(missing) == 0:
            return candidate_dir, layout_name, []

    diagnostic_dir = root / case_id / "mri"
    diagnostic_files = {
        key: diagnostic_dir / filename
        for key, filename in FASTSURFER_REQUIRED_FILES.items()
    }
    missing = [key for key, path in diagnostic_files.items() if not _path_exists_and_valid(path)]
    return None, "not_found", missing


def remap_sample_to_fastsurfer_root(sample, fastsurfer_root, verbose=True):
    """
    Keeps BRATS image/label paths from splits.json, but replaces FastSurfer
    paths using the selected FastSurfer root.
    """
    sample = copy.deepcopy(sample)
    case_id = sample["id"]

    case_mri_dir, layout_name, missing = find_fastsurfer_case_dir(
        case_id,
        fastsurfer_root,
    )

    if case_mri_dir is None:
        if verbose:
            print(
                f"[Missing FastSurfer] {case_id}: no valid FastSurfer files under {fastsurfer_root}. "
                f"Missing under case/mri candidate: {missing}"
            )
        return None

    sample["fastsurfer_mri_dir"] = str(case_mri_dir)
    sample["fastsurfer"] = {
        "dkt": str(case_mri_dir / FASTSURFER_REQUIRED_FILES["dkt"]),
        "aseg": str(case_mri_dir / FASTSURFER_REQUIRED_FILES["aseg"]),
        "mask": str(case_mri_dir / FASTSURFER_REQUIRED_FILES["mask"]),
    }
    sample["fastsurfer_layout"] = layout_name
    sample["fastsurfer_root_used"] = str(fastsurfer_root)

    if verbose:
        has_label = bool(sample.get("seg")) and os.path.exists(sample.get("seg", ""))
        print(
            f"[Matched FastSurfer] {case_id}: layout={layout_name}, "
            f"label_available={has_label}"
        )

    return sample


def build_labeled_eval_samples_from_split(eval_cfg):
    """
    Validation:
        uses split val records as-is.

    Test:
        uses split test records, but remaps FastSurfer paths to outputGLItraining
        because this is the labeled 1251-case FastSurfer output root.
    """
    split_data = load_json(eval_cfg.split_path)

    val_samples = split_data.get("val", [])
    raw_test_samples = split_data.get("test", [])

    print(f"[Splits] Validation cases in split: {len(val_samples)}")
    print(f"[Splits] Test cases in split:       {len(raw_test_samples)}")

    remapped_test_samples = []
    missing_test_samples = []

    print("\n[Labeled test FastSurfer matching using outputGLItraining]")
    for sample in raw_test_samples:
        remapped = remap_sample_to_fastsurfer_root(
            sample,
            eval_cfg.fastsurfer_labeled_root,
            verbose=True,
        )
        if remapped is None:
            missing_test_samples.append(sample["id"])
        else:
            remapped_test_samples.append(remapped)

    print("\n[Labeled test FastSurfer summary]")
    print(f"Test cases in split:              {len(raw_test_samples)}")
    print(f"Test cases with FastSurfer match: {len(remapped_test_samples)}")
    print(f"Test cases missing FastSurfer:    {len(missing_test_samples)}")

    if missing_test_samples:
        print("[Warning] Missing or mismatched labeled test FastSurfer cases:")
        for case_id in missing_test_samples[:50]:
            print(f"  - {case_id}")
        if len(missing_test_samples) > 50:
            print(f"  ... plus {len(missing_test_samples) - 50} more")

    return val_samples, remapped_test_samples, missing_test_samples


class EvalBratsFastSurferDataset(BratsFastSurferDataset):
    """
    Evaluation dataset that supports optional labels.

    For the labeled split test, labels should exist.
    If a label is missing, metrics for that case become NaN.
    """

    def __getitem__(self, idx):
        sample = self.samples[idx]

        t1 = zscore_nonzero(load_volume(sample["mri"]["t1"], dtype=np.float32))
        t1ce = zscore_nonzero(load_volume(sample["mri"]["t1ce"], dtype=np.float32))
        t2 = zscore_nonzero(load_volume(sample["mri"]["t2"], dtype=np.float32))
        flair = zscore_nonzero(load_volume(sample["mri"]["flair"], dtype=np.float32))

        x = np.stack([t1, t1ce, t2, flair], axis=0).astype(np.float32, copy=False)

        seg_path = sample.get("seg", None)
        has_label = bool(seg_path) and os.path.exists(seg_path)
        if has_label:
            seg = remap_brats_labels(load_volume(seg_path, dtype=np.int32)).astype(np.int64)
        else:
            seg = np.zeros(x.shape[1:], dtype=np.int64)

        priors, region = load_or_build_priors(self.cfg, sample)

        if priors.shape[1:] != x.shape[1:] or region.shape != x.shape[1:]:
            print(f"[Warning] Shape mismatch before crop for {sample['id']}:")
            print(f"  MRI shape:     {x.shape[1:]}")
            print(f"  priors shape:  {priors.shape}")
            print(f"  region shape:  {region.shape}")
        else:
            if idx < 3:
                print(
                    f"[Resampling OK] {sample['id']}: "
                    f"MRI={x.shape[1:]}, priors={priors.shape}, region={region.shape}"
                )

        x = self._pad_if_needed(x, self.cfg.patch_size, value=0)
        priors = self._pad_if_needed(priors, self.cfg.patch_size, value=0)
        seg = self._pad_if_needed(seg, self.cfg.patch_size, value=0)
        region = self._pad_if_needed(region, self.cfg.patch_size, value=0)

        x = self._center_crop(x, self.cfg.patch_size)
        priors = self._center_crop(priors, self.cfg.patch_size)
        seg = self._center_crop(seg, self.cfg.patch_size)
        region = self._center_crop(region, self.cfg.patch_size)

        return {
            "id": sample["id"],
            "image": torch.from_numpy(x).float(),
            "priors": torch.from_numpy(priors).float(),
            "region": torch.from_numpy(region).long(),
            "seg": torch.from_numpy(seg).long(),
            "has_label": torch.tensor(bool(has_label), dtype=torch.bool),
        }


def build_labeled_post_training_eval_loaders(eval_cfg):
    """
    Builds validation and labeled test loaders without changing training.

    Validation:
        uses eval_cfg.val_cache_dir.

    Test:
        uses splits["test"], outputGLItraining FastSurfer paths, and isolated
        eval_cfg.test_cache_dir.
    """
    val_samples, test_samples, missing_test_samples = build_labeled_eval_samples_from_split(eval_cfg)

    val_cfg = copy.deepcopy(eval_cfg)
    val_cfg.cache_dir = eval_cfg.val_cache_dir
    val_cfg.use_patch_sampling = False

    test_cfg = copy.deepcopy(eval_cfg)
    test_cfg.cache_dir = eval_cfg.test_cache_dir
    test_cfg.use_patch_sampling = False

    print("\n[DataLoader] Building validation dataset/loader...")
    val_ds = EvalBratsFastSurferDataset(
        cfg=val_cfg,
        samples=val_samples,
        training=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=eval_cfg.num_workers,
        pin_memory=True,
    )

    print("[DataLoader] Building labeled test dataset/loader...")
    test_ds = EvalBratsFastSurferDataset(
        cfg=test_cfg,
        samples=test_samples,
        training=False,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        num_workers=eval_cfg.num_workers,
        pin_memory=True,
    )

    print(f"[DataLoader] Validation cases:   {len(val_ds)}")
    print(f"[DataLoader] Labeled test cases: {len(test_ds)}")

    return val_loader, test_loader, missing_test_samples

In [ ]:
def safe_float(x):
    try:
        if x is None:
            return float("nan")
        return float(x)
    except Exception:
        return float("nan")


def foreground_class_dice_from_arrays(pred, target, num_classes=4, eps=1e-6):
    """
    Matches the notebook validation convention:
    mean Dice over foreground classes 1, 2, and 3.

    pred:   integer array [D, H, W]
    target: integer array [D, H, W]
    """
    class_dice = {}
    values = []

    for cls in range(1, num_classes):
        pred_c = pred == cls
        target_c = target == cls

        pred_sum = pred_c.sum()
        target_sum = target_c.sum()

        if pred_sum == 0 and target_sum == 0:
            dice = 1.0
        else:
            inter = np.logical_and(pred_c, target_c).sum()
            dice = (2.0 * inter + eps) / (pred_sum + target_sum + eps)

        class_dice[f"dice_class_{cls}"] = float(dice)
        values.append(float(dice))

    class_dice["mean_dice"] = float(np.mean(values)) if values else float("nan")
    return class_dice


def binary_foreground_metrics(pred_fg, target_fg, eps=1e-6):
    """
    Binary foreground metrics using consistent empty-mask rules.

    Empty-mask rules:
    - GT empty and prediction empty:
        dice=1, precision=1, recall=1
    - GT empty and prediction non-empty:
        dice=0, precision=0, recall=NaN
    - GT non-empty and prediction empty:
        dice=0, precision=NaN, recall=0
    """
    pred_fg = pred_fg.astype(bool)
    target_fg = target_fg.astype(bool)

    pred_sum = int(pred_fg.sum())
    target_sum = int(target_fg.sum())
    tp = int(np.logical_and(pred_fg, target_fg).sum())
    fp = int(np.logical_and(pred_fg, ~target_fg).sum())
    fn = int(np.logical_and(~pred_fg, target_fg).sum())

    if pred_sum == 0 and target_sum == 0:
        return {
            "binary_dice": 1.0,
            "precision": 1.0,
            "recall": 1.0,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "pred_fg_voxels": pred_sum,
            "target_fg_voxels": target_sum,
        }

    if pred_sum == 0 and target_sum > 0:
        precision = float("nan")
        recall = 0.0
    elif pred_sum > 0 and target_sum == 0:
        precision = 0.0
        recall = float("nan")
    else:
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)

    dice = (2.0 * tp + eps) / (pred_sum + target_sum + eps)

    return {
        "binary_dice": float(dice),
        "precision": safe_float(precision),
        "recall": safe_float(recall),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "pred_fg_voxels": pred_sum,
        "target_fg_voxels": target_sum,
    }


def binary_auroc_auprc(prob_fg, target_fg):
    """
    AUROC/AUPRC for binary foreground probability.

    prob_fg:   float array [D, H, W], p_foreground
    target_fg: bool/int array [D, H, W], foreground target
    """
    if not SKLEARN_AVAILABLE:
        return float("nan"), float("nan"), "sklearn_unavailable"

    y_true = target_fg.astype(np.uint8).reshape(-1)
    y_score = prob_fg.astype(np.float32).reshape(-1)

    unique = np.unique(y_true)
    if unique.size < 2:
        return float("nan"), float("nan"), "single_class_target"

    try:
        auroc = roc_auc_score(y_true, y_score)
    except Exception as exc:
        auroc = float("nan")
        return auroc, float("nan"), f"auroc_error:{repr(exc)}"

    try:
        auprc = average_precision_score(y_true, y_score)
    except Exception as exc:
        auprc = float("nan")
        return auroc, auprc, f"auprc_error:{repr(exc)}"

    return float(auroc), float(auprc), "ok"


def _surface_mask(mask):
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return mask
    eroded = binary_erosion(mask, structure=np.ones((3, 3, 3)), border_value=0)
    return np.logical_xor(mask, eroded)


def hd95_binary(pred_fg, target_fg):
    """
    HD95 on binary foreground masks in voxel units.

    Empty-mask rules:
    - both empty: 0.0
    - one empty: NaN
    """
    if not SCIPY_AVAILABLE:
        return float("nan")

    pred_fg = pred_fg.astype(bool)
    target_fg = target_fg.astype(bool)

    pred_sum = int(pred_fg.sum())
    target_sum = int(target_fg.sum())

    if pred_sum == 0 and target_sum == 0:
        return 0.0

    if pred_sum == 0 or target_sum == 0:
        return float("nan")

    pred_surface = _surface_mask(pred_fg)
    target_surface = _surface_mask(target_fg)

    if pred_surface.sum() == 0 or target_surface.sum() == 0:
        return float("nan")

    # Distance from every voxel to the nearest surface voxel of the other mask.
    dt_pred = distance_transform_edt(~pred_surface)
    dt_target = distance_transform_edt(~target_surface)

    distances_pred_to_target = dt_target[pred_surface]
    distances_target_to_pred = dt_pred[target_surface]

    all_distances = np.concatenate([distances_pred_to_target, distances_target_to_pred])
    if all_distances.size == 0:
        return float("nan")

    return float(np.percentile(all_distances, 95))


def summarize_nanmean(values):
    arr = np.asarray(values, dtype=np.float64)
    if arr.size == 0 or np.all(np.isnan(arr)):
        return float("nan")
    return float(np.nanmean(arr))


def extract_case_id_from_batch(batch):
    case_id = batch.get("id", ["unknown"])
    if isinstance(case_id, (list, tuple)):
        return str(case_id[0])
    return str(case_id)

In [ ]:
def safe_torch_load_checkpoint(path, map_location="cpu"):
    """
    Compatible with older/newer PyTorch versions.
    """
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def build_acr_model_for_evaluation(eval_cfg, device):
    """
    Instantiates the same acf_net-base architecture used during training.
    """
    model = ACRSWAFNet(
        num_classes=eval_cfg.num_classes,
        num_prior_channels=9,
        num_regions=eval_cfg.num_regions,
        region_vocab=eval_cfg.region_vocab,
        router_hidden=eval_cfg.router_hidden,
        router_region_dim=eval_cfg.router_region_dim,
        router_global_dim=eval_cfg.router_global_dim,
        router_calibration_lambda=eval_cfg.router_calibration_lambda,
        debug_shapes=False,
        return_voxel_weights=False,
    ).to(device)

    return model


def load_best_acr_checkpoint_for_eval(eval_cfg, device=None):
    """
    Loads only:
        FILL IN WITH DIRECTORY FOR BEST MODEL CHECKPOINT/best_model.pt

    It does not search original reference model directories.
    """
    if device is None:
        device = eval_cfg.device

    ckpt_path = eval_cfg.best_checkpoint_path

    print("=" * 80)
    print("Loading best acf_net-base checkpoint for final evaluation")
    print("=" * 80)
    print(f"Checkpoint path: {ckpt_path}")

    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Best ACR checkpoint not found: {ckpt_path}")

    model = build_acr_model_for_evaluation(eval_cfg, device)
    ckpt = safe_torch_load_checkpoint(ckpt_path, map_location=device)

    if "model_state" not in ckpt:
        raise RuntimeError(f"Checkpoint does not contain 'model_state': {ckpt_path}")

    try:
        model.load_state_dict(ckpt["model_state"], strict=True)
    except RuntimeError as exc:
        print("[Checkpoint error] The checkpoint is incompatible with the current ACRSWAFNet definition.")
        print(str(exc))
        raise

    model.eval()

    best_epoch = int(ckpt.get("epoch", -1))
    best_metric = safe_float(ckpt.get("best_metric", float("nan")))

    print("[Checkpoint] Load successful.")
    print(f"[Checkpoint] best_epoch:  {best_epoch}")
    print(f"[Checkpoint] best_metric: {best_metric:.6f}")

    if "history" in ckpt and isinstance(ckpt["history"], list):
        print(f"[Checkpoint] history rows: {len(ckpt['history'])}")

    return model, ckpt, best_epoch, best_metric

In [ ]:
@torch.no_grad()
def evaluate_validation_set_acr(model, val_loader, eval_cfg):
    """
    Computes per-case validation Dice using the existing notebook convention:
    mean Dice over foreground classes 1, 2, and 3.
    """
    model.eval()
    rows = []

    print("=" * 80)
    print("Evaluating validation set")
    print("=" * 80)

    for batch in tqdm(val_loader, desc="Validation evaluation", leave=True):
        case_id = extract_case_id_from_batch(batch)

        image = batch["image"].to(eval_cfg.device, non_blocking=True)
        priors = batch["priors"].to(eval_cfg.device, non_blocking=True)
        region = batch["region"].to(eval_cfg.device, non_blocking=True)
        target = batch["seg"].to(eval_cfg.device, non_blocking=True)
        has_label = bool(batch.get("has_label", torch.tensor([True]))[0].item())

        if not has_label:
            rows.append({
                "case_id": case_id,
                "has_label": False,
                "mean_dice": float("nan"),
                "dice_class_1": float("nan"),
                "dice_class_2": float("nan"),
                "dice_class_3": float("nan"),
            })
            continue

        outputs = model(image, priors, region)
        logits = outputs["logits_fused"]
        pred = torch.argmax(logits, dim=1)

        pred_np = pred[0].detach().cpu().numpy().astype(np.int16)
        target_np = target[0].detach().cpu().numpy().astype(np.int16)

        dice = foreground_class_dice_from_arrays(
            pred_np,
            target_np,
            num_classes=eval_cfg.num_classes,
        )

        rows.append({
            "case_id": case_id,
            "has_label": True,
            "mean_dice": dice["mean_dice"],
            "dice_class_1": dice.get("dice_class_1", float("nan")),
            "dice_class_2": dice.get("dice_class_2", float("nan")),
            "dice_class_3": dice.get("dice_class_3", float("nan")),
        })

    val_df = pd.DataFrame(rows)
    mean_val_dice = summarize_nanmean(val_df["mean_dice"].values) if len(val_df) else float("nan")

    print(f"[Validation] cases evaluated: {len(val_df)}")
    print(f"[Validation] mean_val_dice: {mean_val_dice:.6f}")

    return val_df, mean_val_dice

In [ ]:
@torch.no_grad()
def evaluate_test_set_acr(model, test_loader, eval_cfg):
    """
    Test metrics:
    - mean_test_dice: foreground-class mean Dice over classes 1, 2, 3
    - mean_test_auroc: binary foreground AUROC using p_foreground = 1 - p_background
    - mean_test_auprc: binary foreground AUPRC using p_foreground = 1 - p_background
    - mean_test_precision: binary foreground precision
    - mean_test_recall: binary foreground recall
    - mean_test_hausdorff: HD95 on binary foreground masks in voxel units
    """
    model.eval()
    rows = []

    print("=" * 80)
    print("Evaluating test set")
    print("=" * 80)
    print("[Metric note] mean_test_hausdorff is HD95 on binary foreground masks in voxel units.")

    if len(test_loader.dataset) == 0:
        print("[Warning] Test loader has 0 cases. No test metrics will be computed.")
        return pd.DataFrame(rows)

    for batch in tqdm(test_loader, desc="Test evaluation", leave=True):
        case_id = extract_case_id_from_batch(batch)

        image = batch["image"].to(eval_cfg.device, non_blocking=True)
        priors = batch["priors"].to(eval_cfg.device, non_blocking=True)
        region = batch["region"].to(eval_cfg.device, non_blocking=True)
        target = batch["seg"].to(eval_cfg.device, non_blocking=True)
        has_label = bool(batch.get("has_label", torch.tensor([True]))[0].item())

        outputs = model(image, priors, region)
        logits = outputs["logits_fused"]
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(logits, dim=1)

        pred_np = pred[0].detach().cpu().numpy().astype(np.int16)
        prob_fg_np = (1.0 - probs[0, 0].detach().cpu().numpy()).astype(np.float32)

        row = {
            "case_id": case_id,
            "has_label": has_label,
            "auroc": float("nan"),
            "auprc": float("nan"),
            "auroc_auprc_status": "label_missing",
            "mean_dice": float("nan"),
            "dice_class_1": float("nan"),
            "dice_class_2": float("nan"),
            "dice_class_3": float("nan"),
            "binary_dice": float("nan"),
            "precision": float("nan"),
            "recall": float("nan"),
            "hausdorff_hd95": float("nan"),
            "pred_fg_voxels": int((pred_np > 0).sum()),
            "target_fg_voxels": float("nan"),
            "tp": float("nan"),
            "fp": float("nan"),
            "fn": float("nan"),
        }

        if not has_label:
            print(f"[Labels missing] {case_id}: label-dependent metrics saved as NaN.")
            rows.append(row)
            continue

        target_np = target[0].detach().cpu().numpy().astype(np.int16)

        pred_fg = pred_np > 0
        target_fg = target_np > 0

        # Class-wise foreground mean Dice, matching validation convention.
        dice = foreground_class_dice_from_arrays(
            pred_np,
            target_np,
            num_classes=eval_cfg.num_classes,
        )

        # Binary precision/recall/Dice.
        bin_metrics = binary_foreground_metrics(pred_fg, target_fg)

        # Binary AUROC/AUPRC from probabilities.
        auroc, auprc, status = binary_auroc_auprc(prob_fg_np, target_fg)

        # HD95 on binary foreground masks.
        hd95 = hd95_binary(pred_fg, target_fg)

        row.update({
            "auroc": safe_float(auroc),
            "auprc": safe_float(auprc),
            "auroc_auprc_status": status,
            "mean_dice": dice["mean_dice"],
            "dice_class_1": dice.get("dice_class_1", float("nan")),
            "dice_class_2": dice.get("dice_class_2", float("nan")),
            "dice_class_3": dice.get("dice_class_3", float("nan")),
            "binary_dice": bin_metrics["binary_dice"],
            "precision": bin_metrics["precision"],
            "recall": bin_metrics["recall"],
            "hausdorff_hd95": safe_float(hd95),
            "pred_fg_voxels": bin_metrics["pred_fg_voxels"],
            "target_fg_voxels": bin_metrics["target_fg_voxels"],
            "tp": bin_metrics["tp"],
            "fp": bin_metrics["fp"],
            "fn": bin_metrics["fn"],
        })

        rows.append(row)

    test_df = pd.DataFrame(rows)

    print(f"[Test] cases evaluated: {len(test_df)}")
    if len(test_df):
        print(f"[Test] labels available: {int(test_df['has_label'].sum())}/{len(test_df)}")
        if int(test_df["has_label"].sum()) == 0:
            print("[Warning] No test labels were available. All label-dependent test metrics are NaN.")

    return test_df

In [ ]:
def create_and_save_acr_final_results(
    eval_cfg,
    best_epoch,
    best_metric,
    val_df,
    test_df,
    missing_test_samples=None,
):
    """
    Creates and saves:
    - summary_results_acf_net_base.csv
    - summary_results_acf_net_base.json
    - per_case_test_results_acf_net_base.csv
    - per_case_val_results_acf_net_base.csv

    Official convention:
    mean_val_dice = ckpt["best_metric"]
    """
    missing_test_samples = missing_test_samples or []
    ensure_dir(eval_cfg.results_dir)

    per_case_val_path = os.path.join(eval_cfg.results_dir, "per_case_val_results_acf_net_base.csv")
    per_case_test_path = os.path.join(eval_cfg.results_dir, "per_case_test_results_acf_net_base.csv")
    summary_csv_path = os.path.join(eval_cfg.results_dir, "summary_results_acf_net_base.csv")
    summary_json_path = os.path.join(eval_cfg.results_dir, "summary_results_acf_net_base.json")

    if val_df is not None and len(val_df):
        val_df.to_csv(per_case_val_path, index=False)
        recomputed_val_dice = summarize_nanmean(val_df["mean_dice"].values)
    else:
        recomputed_val_dice = float("nan")

    official_mean_val_dice = safe_float(best_metric)

    if test_df is not None and len(test_df):
        test_df.to_csv(per_case_test_path, index=False)

        mean_test_auroc = summarize_nanmean(test_df["auroc"].values)
        mean_test_auprc = summarize_nanmean(test_df["auprc"].values)
        mean_test_dice = summarize_nanmean(test_df["mean_dice"].values)
        mean_test_precision = summarize_nanmean(test_df["precision"].values)
        mean_test_recall = summarize_nanmean(test_df["recall"].values)
        mean_test_hausdorff = summarize_nanmean(test_df["hausdorff_hd95"].values)

        labels_available = int(test_df["has_label"].sum()) if "has_label" in test_df.columns else 0
        total_test_cases = len(test_df)
    else:
        mean_test_auroc = float("nan")
        mean_test_auprc = float("nan")
        mean_test_dice = float("nan")
        mean_test_precision = float("nan")
        mean_test_recall = float("nan")
        mean_test_hausdorff = float("nan")
        labels_available = 0
        total_test_cases = 0

    summary_row = {
        "model": "acf_net-base",
        "best_epoch": int(best_epoch),

        # Official validation convention
        "mean_val_dice": official_mean_val_dice,
        "checkpoint_best_metric": official_mean_val_dice,
        "recomputed_val_dice_sanity_check": safe_float(recomputed_val_dice),

        # Final held-out test metrics
        "mean_test_auroc": safe_float(mean_test_auroc),
        "mean_test_auprc": safe_float(mean_test_auprc),
        "mean_test_dice": safe_float(mean_test_dice),
        "mean_test_precision": safe_float(mean_test_precision),
        "mean_test_recall": safe_float(mean_test_recall),
        "mean_test_hausdorff": safe_float(mean_test_hausdorff),

        "hausdorff_definition": "HD95 on binary foreground masks in voxel units",
        "validation_definition": "best validation Dice stored in best_model.pt as ckpt['best_metric']",
        "test_definition": "held-out splits['test'] evaluated once after training",
        "test_fastsurfer_root_used": eval_cfg.fastsurfer_labeled_root,
        "external_fastsurfer_root_not_used_for_labeled_metrics": eval_cfg.fastsurfer_external_root,

        "test_cases_evaluated": int(total_test_cases),
        "test_cases_with_labels": int(labels_available),
        "test_cases_missing_fastsurfer": int(len(missing_test_samples)),

        "best_checkpoint_path": eval_cfg.best_checkpoint_path,
        "split_path": eval_cfg.split_path,
        "test_cache_dir": eval_cfg.test_cache_dir,
    }

    summary_df = pd.DataFrame([summary_row])
    summary_df.to_csv(summary_csv_path, index=False)

    with open(summary_json_path, "w") as f:
        json.dump(
            {
                "summary": summary_row,
                "missing_test_fastsurfer_cases": missing_test_samples,
            },
            f,
            indent=2,
        )

    print("=" * 80)
    print("Final acf_net-base results")
    print("=" * 80)
    print("[Validation note] mean_val_dice is ckpt['best_metric'] from best_model.pt.")
    print("[Test note] Test metrics are computed on held-out splits['test'].")
    print("[Anatomy note] Labeled test metrics use outputGLItraining FastSurfer priors.")
    print("[Hausdorff note] mean_test_hausdorff is HD95 on binary foreground masks in voxel units.")
    display(summary_df)

    print("\n[Saved result files]")
    if val_df is not None and len(val_df):
        print(f"Per-case validation CSV: {per_case_val_path}")
    else:
        print("Per-case validation CSV: not saved because no validation rows were available")

    if test_df is not None and len(test_df):
        print(f"Per-case test CSV:       {per_case_test_path}")
    else:
        print("Per-case test CSV: not saved because no test rows were available")

    print(f"Summary CSV:             {summary_csv_path}")
    print(f"Summary JSON:            {summary_json_path}")

    if labels_available == 0:
        print(
            "\n[Warning] No test labels were available, so AUROC, AUPRC, Dice, "
            "precision, recall, and HD95 are NaN."
        )

    if missing_test_samples:
        print(
            f"\n[Warning] {len(missing_test_samples)} test cases were skipped because "
            "matching outputGLItraining FastSurfer files were missing or mismatched."
        )

    return summary_df, summary_row

In [ ]:
print("=" * 80)
print("Running acf_net-base final labeled evaluation")
print("=" * 80)

assert eval_cfg.best_checkpoint_path == (
    "FILL IN WITH DIRECTORY FOR BEST MODEL CHECKPOINT/best_model.pt"
), eval_cfg.best_checkpoint_path

assert "FILL IN WITH DIRECTORY FOR ACF NET BASE BRATS RUNS/" in eval_cfg.best_checkpoint_path, (
    "Refusing to run because best checkpoint path is not inside the isolated acf_net-base run directory."
)

assert os.path.exists(eval_cfg.best_checkpoint_path), (
    f"Missing best checkpoint: {eval_cfg.best_checkpoint_path}"
)

# Build loaders from existing split file.
# Test uses splits["test"] + outputGLItraining FastSurfer priors.
val_loader_eval, test_loader_eval, missing_test_samples = build_labeled_post_training_eval_loaders(eval_cfg)

print("\n[Checkpoint] Loading best ACR checkpoint...")
acr_eval_model, acr_ckpt, best_epoch, best_metric = load_best_acr_checkpoint_for_eval(
    eval_cfg,
    device=eval_cfg.device,
)

print("\n[Validation] Optional sanity-check evaluation on splits['val']...")
print("[Validation note] Official mean_val_dice in the final table will use ckpt['best_metric'].")
per_case_val_df, recomputed_mean_val_dice = evaluate_validation_set_acr(
    acr_eval_model,
    val_loader_eval,
    eval_cfg,
)

print("\n[Test] Final held-out evaluation on splits['test']...")
per_case_test_df = evaluate_test_set_acr(
    acr_eval_model,
    test_loader_eval,
    eval_cfg,
)

print("\n[Results] Creating final summary table and saving files...")
summary_results_df, summary_results_row = create_and_save_acr_final_results(
    eval_cfg=eval_cfg,
    best_epoch=best_epoch,
    best_metric=best_metric,
    val_df=per_case_val_df,
    test_df=per_case_test_df,
    missing_test_samples=missing_test_samples,
)

print("\nDone.")

In [ ]:
from pathlib import Path
import os
import json
import copy
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import nibabel as nib
    NIBABEL_AVAILABLE = True
except Exception as exc:
    NIBABEL_AVAILABLE = False
    print(f"[Warning] nibabel unavailable: {repr(exc)}")

try:
    from sklearn.metrics import roc_auc_score, average_precision_score
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    print(f"[Warning] sklearn metrics unavailable: {repr(exc)}")

try:
    from scipy.ndimage import binary_erosion, distance_transform_edt
    SCIPY_AVAILABLE = True
except Exception as exc:
    SCIPY_AVAILABLE = False
    print(f"[Warning] scipy HD95 helpers unavailable: {repr(exc)}")

eval_cfg = TrainConfig()

# ACR-only paths. Do not point these to original reference model directories.
eval_cfg.work_dir = "FILL IN WITH DIRECTORY FOR ACF NET BASE BRATS RUNS"
eval_cfg.checkpoint_dir = "FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS"
eval_cfg.results_dir = "FILL IN WITH DIRECTORY FOR EVALUATION RESULTS"
eval_cfg.split_path = "FILL IN WITH DIRECTORY FOR TRAIN VALIDATION TEST SPLITS/splits.json"
eval_cfg.best_checkpoint_path = "FILL IN WITH DIRECTORY FOR BEST MODEL CHECKPOINT/best_model.pt"

# External test FastSurfer path.
eval_cfg.fastsurfer_outputglitest_root = "FILL IN WITH DIRECTORY FOR FASTSURFER TEST OUTPUTS"

# Labeled split-overlap FastSurfer path from your consistency rule.
eval_cfg.fastsurfer_labeled_root = "FILL IN WITH DIRECTORY FOR FASTSURFER LABELED TRAINING OUTPUTS"

# Isolated prior caches.
eval_cfg.val_cache_dir = "FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE"
eval_cfg.test_cache_dir = "FILL IN WITH DIRECTORY FOR TEST PRIOR CACHE"

# Evaluation behavior.
eval_cfg.batch_size = 1
eval_cfg.num_workers = 2
eval_cfg.use_patch_sampling = False
eval_cfg.DEBUG_SHAPES = False
eval_cfg.debug_resampling_shapes = False

# If outputGLItest has no overlap with splits["test"], fall back to outputGLItraining.
AUTO_FALLBACK_TO_LABELED_FASTSURFER_ROOT = True

ensure_dir(eval_cfg.results_dir)
ensure_dir(eval_cfg.test_cache_dir)

print("=" * 80)
print("acf_net-base BraTS-region post-training evaluation config")
print("=" * 80)
print(f"Best checkpoint:       {eval_cfg.best_checkpoint_path}")
print(f"Split file:            {eval_cfg.split_path}")
print(f"Results dir:           {eval_cfg.results_dir}")
print(f"Val prior cache:       {eval_cfg.val_cache_dir}")
print(f"Test prior cache:      {eval_cfg.test_cache_dir}")
print(f"External FS root:      {eval_cfg.fastsurfer_outputglitest_root}")
print(f"Labeled fallback root: {eval_cfg.fastsurfer_labeled_root}")

assert os.path.exists(eval_cfg.best_checkpoint_path), f"Missing checkpoint: {eval_cfg.best_checkpoint_path}"
assert os.path.exists(eval_cfg.split_path), f"Missing split file: {eval_cfg.split_path}"
assert os.path.exists(eval_cfg.fastsurfer_outputglitest_root), f"Missing outputGLItest root: {eval_cfg.fastsurfer_outputglitest_root}"

print("\n[Label convention]")
print("Assumed remapped labels: 0=background, 1=tumor core/non-enhancing, 2=edema, 3=enhancing tumor")
print("WT = {1,2,3}, TC = {1,3}, ET = {3}")
print("HD95 rule: both empty -> 0.0; one empty -> NaN")

In [ ]:
FASTSURFER_REQUIRED_FILES = {
    "dkt": "aparc.DKTatlas+aseg.deep.mgz",
    "aseg": "aseg.auto_noCCseg.mgz",
    "mask": "mask.mgz",
}


def _valid_file(path_obj):
    path_obj = Path(path_obj)
    return path_obj.exists() and path_obj.is_file() and not str(path_obj).endswith(":Zone.Identifier")


def find_fastsurfer_case_dir(case_id, fastsurfer_root):
    root = Path(fastsurfer_root)
    candidates = [
        (root / case_id / "mri", "case/mri"),
        (root / case_id, "case_root"),
    ]

    for candidate_dir, layout in candidates:
        files = {
            key: candidate_dir / filename
            for key, filename in FASTSURFER_REQUIRED_FILES.items()
        }
        missing = [key for key, path in files.items() if not _valid_file(path)]
        if len(missing) == 0:
            return candidate_dir, layout, []

    diagnostic_dir = root / case_id / "mri"
    missing = [
        key
        for key, filename in FASTSURFER_REQUIRED_FILES.items()
        if not _valid_file(diagnostic_dir / filename)
    ]
    return None, "not_found", missing


def remap_sample_fastsurfer(sample, fastsurfer_root, verbose=False):
    sample = copy.deepcopy(sample)
    case_id = sample["id"]

    case_dir, layout, missing = find_fastsurfer_case_dir(case_id, fastsurfer_root)
    if case_dir is None:
        if verbose:
            print(f"[Missing FastSurfer] {case_id}: missing={missing}")
        return None

    sample["fastsurfer_mri_dir"] = str(case_dir)
    sample["fastsurfer"] = {
        "dkt": str(case_dir / FASTSURFER_REQUIRED_FILES["dkt"]),
        "aseg": str(case_dir / FASTSURFER_REQUIRED_FILES["aseg"]),
        "mask": str(case_dir / FASTSURFER_REQUIRED_FILES["mask"]),
    }
    sample["fastsurfer_layout"] = layout
    sample["fastsurfer_root_used"] = str(fastsurfer_root)

    if verbose:
        has_label = bool(sample.get("seg")) and os.path.exists(sample.get("seg", ""))
        print(f"[Matched] {case_id}: layout={layout}, label_available={has_label}")

    return sample


def count_fastsurfer_matches(samples, fastsurfer_root, max_print=10):
    matched = []
    missing = []

    for sample in samples:
        remapped = remap_sample_fastsurfer(sample, fastsurfer_root, verbose=False)
        if remapped is None:
            missing.append(sample["id"])
        else:
            matched.append(remapped)

    print(f"[FastSurfer check] root={fastsurfer_root}")
    print(f"  matched: {len(matched)}")
    print(f"  missing: {len(missing)}")

    if missing:
        print("  first missing cases:")
        for case_id in missing[:max_print]:
            print(f"    - {case_id}")
        if len(missing) > max_print:
            print(f"    ... plus {len(missing) - max_print} more")

    return matched, missing


def build_brats_region_eval_samples(eval_cfg):
    split_data = load_json(eval_cfg.split_path)

    val_samples = split_data.get("val", [])
    raw_test_samples = split_data.get("test", [])

    print("=" * 80)
    print("Loading evaluation splits")
    print("=" * 80)
    print(f"Validation cases: {len(val_samples)}")
    print(f"Test cases:       {len(raw_test_samples)}")

    print("\n[Test FastSurfer matching: external outputGLItest]")
    test_samples, missing_test = count_fastsurfer_matches(
        raw_test_samples,
        eval_cfg.fastsurfer_outputglitest_root,
    )
    selected_test_root = eval_cfg.fastsurfer_outputglitest_root

    if len(test_samples) == 0 and AUTO_FALLBACK_TO_LABELED_FASTSURFER_ROOT:
        print("\n[Warning] outputGLItest has zero overlap with splits['test'].")
        print("[Fallback] Switching test FastSurfer root to outputGLItraining for labeled held-out metrics.")
        test_samples, missing_test = count_fastsurfer_matches(
            raw_test_samples,
            eval_cfg.fastsurfer_labeled_root,
        )
        selected_test_root = eval_cfg.fastsurfer_labeled_root

    print("\n[Selected test FastSurfer root]")
    print(selected_test_root)
    print(f"Test cases with matching FastSurfer outputs: {len(test_samples)}")
    print(f"Test cases missing FastSurfer outputs:       {len(missing_test)}")

    return val_samples, test_samples, missing_test, selected_test_root


def get_nifti_spacing(path):
    if not NIBABEL_AVAILABLE:
        return (1.0, 1.0, 1.0)

    try:
        img = nib.load(str(path))
        zooms = img.header.get_zooms()
        if len(zooms) >= 3:
            return tuple(float(z) for z in zooms[:3])
    except Exception:
        pass

    return (1.0, 1.0, 1.0)


class BratsRegionEvalDataset(BratsFastSurferDataset):
    """
    Evaluation-only dataset.

    Returns:
    - image, priors, region, seg
    - has_label
    - spacing from T1 NIfTI header when available
    """

    def __getitem__(self, idx):
        sample = self.samples[idx]

        t1 = zscore_nonzero(load_volume(sample["mri"]["t1"], dtype=np.float32))
        t1ce = zscore_nonzero(load_volume(sample["mri"]["t1ce"], dtype=np.float32))
        t2 = zscore_nonzero(load_volume(sample["mri"]["t2"], dtype=np.float32))
        flair = zscore_nonzero(load_volume(sample["mri"]["flair"], dtype=np.float32))

        x = np.stack([t1, t1ce, t2, flair], axis=0).astype(np.float32, copy=False)

        seg_path = sample.get("seg", None)
        has_label = bool(seg_path) and os.path.exists(seg_path)
        if has_label:
            seg = remap_brats_labels(load_volume(seg_path, dtype=np.int32)).astype(np.int64)
        else:
            seg = np.zeros(x.shape[1:], dtype=np.int64)

        priors, region = load_or_build_priors(self.cfg, sample)

        if idx < 3:
            print(
                f"[Shape check] {sample['id']}: "
                f"MRI={x.shape[1:]}, priors={priors.shape}, region={region.shape}, "
                f"label={has_label}"
            )

        x = self._pad_if_needed(x, self.cfg.patch_size, value=0)
        priors = self._pad_if_needed(priors, self.cfg.patch_size, value=0)
        seg = self._pad_if_needed(seg, self.cfg.patch_size, value=0)
        region = self._pad_if_needed(region, self.cfg.patch_size, value=0)

        x = self._center_crop(x, self.cfg.patch_size)
        priors = self._center_crop(priors, self.cfg.patch_size)
        seg = self._center_crop(seg, self.cfg.patch_size)
        region = self._center_crop(region, self.cfg.patch_size)

        spacing = get_nifti_spacing(sample["mri"]["t1"])

        return {
            "id": sample["id"],
            "image": torch.from_numpy(x).float(),
            "priors": torch.from_numpy(priors).float(),
            "region": torch.from_numpy(region).long(),
            "seg": torch.from_numpy(seg).long(),
            "has_label": torch.tensor(bool(has_label), dtype=torch.bool),
            "spacing": torch.tensor(spacing, dtype=torch.float32),
        }


def build_brats_region_eval_loaders(eval_cfg):
    val_samples, test_samples, missing_test, selected_test_root = build_brats_region_eval_samples(eval_cfg)

    val_cfg = copy.deepcopy(eval_cfg)
    val_cfg.cache_dir = eval_cfg.val_cache_dir
    val_cfg.use_patch_sampling = False

    test_cfg = copy.deepcopy(eval_cfg)
    test_cfg.cache_dir = eval_cfg.test_cache_dir
    test_cfg.use_patch_sampling = False

    print("\n[DataLoader] Building validation loader...")
    val_ds = BratsRegionEvalDataset(cfg=val_cfg, samples=val_samples, training=False)
    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=eval_cfg.num_workers,
        pin_memory=True,
    )

    print("[DataLoader] Building test loader...")
    test_ds = BratsRegionEvalDataset(cfg=test_cfg, samples=test_samples, training=False)
    test_loader = DataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        num_workers=eval_cfg.num_workers,
        pin_memory=True,
    )

    print(f"[DataLoader] Validation cases evaluated: {len(val_ds)}")
    print(f"[DataLoader] Test cases evaluated:       {len(test_ds)}")

    return val_loader, test_loader, missing_test, selected_test_root

In [ ]:
BRATS_REGIONS = {
    "wt": (1, 2, 3),
    "tc": (1, 3),
    "et": (3,),
}


def safe_float(x):
    try:
        if x is None:
            return float("nan")
        return float(x)
    except Exception:
        return float("nan")


def nanmean(values):
    arr = np.asarray(values, dtype=np.float64)
    if arr.size == 0 or np.all(np.isnan(arr)):
        return float("nan")
    return float(np.nanmean(arr))


def mask_from_labels(arr, labels):
    mask = np.zeros_like(arr, dtype=bool)
    for label in labels:
        mask |= arr == label
    return mask


def dice_binary(pred_mask, target_mask, eps=1e-6):
    pred_mask = pred_mask.astype(bool)
    target_mask = target_mask.astype(bool)

    pred_sum = int(pred_mask.sum())
    target_sum = int(target_mask.sum())

    if pred_sum == 0 and target_sum == 0:
        return 1.0
    if pred_sum == 0 and target_sum > 0:
        return 0.0
    if pred_sum > 0 and target_sum == 0:
        return 0.0

    inter = int(np.logical_and(pred_mask, target_mask).sum())
    return float((2.0 * inter + eps) / (pred_sum + target_sum + eps))


def _surface(mask):
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return mask
    eroded = binary_erosion(mask, structure=np.ones((3, 3, 3)), border_value=0)
    return np.logical_xor(mask, eroded)


def hd95_binary(pred_mask, target_mask, spacing=(1.0, 1.0, 1.0)):
    """
    HD95 empty-mask rule:
    - GT empty and prediction empty: 0.0
    - GT non-empty and prediction empty: NaN
    - GT empty and prediction non-empty: NaN
    """
    if not SCIPY_AVAILABLE:
        return float("nan")

    pred_mask = pred_mask.astype(bool)
    target_mask = target_mask.astype(bool)

    pred_sum = int(pred_mask.sum())
    target_sum = int(target_mask.sum())

    if pred_sum == 0 and target_sum == 0:
        return 0.0
    if pred_sum == 0 or target_sum == 0:
        return float("nan")

    pred_surface = _surface(pred_mask)
    target_surface = _surface(target_mask)

    if pred_surface.sum() == 0 or target_surface.sum() == 0:
        return float("nan")

    dt_pred = distance_transform_edt(~pred_surface, sampling=spacing)
    dt_target = distance_transform_edt(~target_surface, sampling=spacing)

    d_pred_to_target = dt_target[pred_surface]
    d_target_to_pred = dt_pred[target_surface]

    distances = np.concatenate([d_pred_to_target, d_target_to_pred])
    if distances.size == 0:
        return float("nan")

    return float(np.percentile(distances, 95))


def classwise_dice(pred, target, num_classes=4):
    out = {}
    vals = []

    for cls in range(1, num_classes):
        d = dice_binary(pred == cls, target == cls)
        out[f"dice_class_{cls}"] = d
        vals.append(d)

    out["mean_foreground_class_dice"] = float(np.mean(vals)) if vals else float("nan")
    return out


def brats_region_metrics(pred, target, spacing=(1.0, 1.0, 1.0), prefix=""):
    out = {}

    dice_vals = []
    hd95_vals = []

    for region_name, labels in BRATS_REGIONS.items():
        pred_mask = mask_from_labels(pred, labels)
        target_mask = mask_from_labels(target, labels)

        dice = dice_binary(pred_mask, target_mask)
        hd95 = hd95_binary(pred_mask, target_mask, spacing=spacing)

        out[f"{prefix}dice_{region_name}"] = dice
        out[f"{prefix}hd95_{region_name}"] = hd95

        dice_vals.append(dice)
        hd95_vals.append(hd95)

    out[f"{prefix}mean_region_dice"] = float(np.mean(dice_vals)) if dice_vals else float("nan")
    out[f"{prefix}mean_hd95"] = nanmean(hd95_vals)
    return out


def wt_binary_probability_from_probs(probs_np):
    """
    WT probability = P(label in {1,2,3}) = 1 - P(background).
    probs_np shape: [C, D, H, W]
    """
    return 1.0 - probs_np[0]


def binary_wt_metrics(pred, target, probs_np):
    pred_wt = pred > 0
    target_wt = target > 0

    tp = int(np.logical_and(pred_wt, target_wt).sum())
    fp = int(np.logical_and(pred_wt, ~target_wt).sum())
    fn = int(np.logical_and(~pred_wt, target_wt).sum())

    pred_sum = int(pred_wt.sum())
    target_sum = int(target_wt.sum())

    if pred_sum == 0 and target_sum == 0:
        precision = 1.0
        recall = 1.0
    elif pred_sum == 0 and target_sum > 0:
        precision = float("nan")
        recall = 0.0
    elif pred_sum > 0 and target_sum == 0:
        precision = 0.0
        recall = float("nan")
    else:
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)

    auroc = float("nan")
    auprc = float("nan")
    au_status = "not_computed"

    if SKLEARN_AVAILABLE:
        y_true = target_wt.astype(np.uint8).reshape(-1)
        y_score = wt_binary_probability_from_probs(probs_np).astype(np.float32).reshape(-1)

        if np.unique(y_true).size >= 2:
            try:
                auroc = float(roc_auc_score(y_true, y_score))
                auprc = float(average_precision_score(y_true, y_score))
                au_status = "ok"
            except Exception as exc:
                au_status = f"metric_error:{repr(exc)}"
        else:
            au_status = "single_class_target"

    return {
        "auroc_wt": safe_float(auroc),
        "auprc_wt": safe_float(auprc),
        "precision_wt": safe_float(precision),
        "recall_wt": safe_float(recall),
        "tp_wt": tp,
        "fp_wt": fp,
        "fn_wt": fn,
        "pred_wt_voxels": pred_sum,
        "target_wt_voxels": target_sum,
        "auroc_auprc_status": au_status,
    }


def batch_spacing_to_tuple(batch):
    spacing = batch.get("spacing", None)
    if spacing is None:
        return (1.0, 1.0, 1.0)

    if isinstance(spacing, torch.Tensor):
        spacing_np = spacing.detach().cpu().numpy()
        spacing_np = np.asarray(spacing_np).reshape(-1)
        if spacing_np.size >= 3:
            return tuple(float(x) for x in spacing_np[:3])

    return (1.0, 1.0, 1.0)


def extract_case_id(batch):
    case_id = batch.get("id", "unknown")
    if isinstance(case_id, (list, tuple)):
        return str(case_id[0])
    return str(case_id)


print("[BraTS region definitions]")
print("WT = labels {1,2,3}")
print("TC = labels {1,3}")
print("ET = label {3}")
print("[HD95] Uses NIfTI spacing when available; otherwise voxel spacing (1,1,1).")

In [ ]:
def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def build_acr_eval_model(eval_cfg, device):
    model = ACRSWAFNet(
        num_classes=eval_cfg.num_classes,
        num_prior_channels=9,
        num_regions=eval_cfg.num_regions,
        region_vocab=eval_cfg.region_vocab,
        router_hidden=eval_cfg.router_hidden,
        router_region_dim=eval_cfg.router_region_dim,
        router_global_dim=eval_cfg.router_global_dim,
        router_calibration_lambda=eval_cfg.router_calibration_lambda,
        debug_shapes=False,
        return_voxel_weights=False,
    ).to(device)
    return model


def load_best_acr_model(eval_cfg):
    print("=" * 80)
    print("Loading best acf_net-base checkpoint")
    print("=" * 80)
    print(eval_cfg.best_checkpoint_path)

    ckpt = safe_torch_load(eval_cfg.best_checkpoint_path, map_location=eval_cfg.device)
    model = build_acr_eval_model(eval_cfg, eval_cfg.device)

    if "model_state" not in ckpt:
        raise RuntimeError("Checkpoint does not contain key: model_state")

    try:
        model.load_state_dict(ckpt["model_state"], strict=True)
    except RuntimeError as exc:
        print("[Error] Checkpoint is incompatible with current ACRSWAFNet definition.")
        print(str(exc))
        raise

    model.eval()

    best_epoch = int(ckpt.get("epoch", -1))
    best_metric = safe_float(ckpt.get("best_metric", float("nan")))

    print("[Checkpoint] Loaded successfully.")
    print(f"[Checkpoint] best_epoch: {best_epoch}")
    print(f"[Checkpoint] ckpt['best_metric']: {best_metric:.6f}")

    return model, ckpt, best_epoch, best_metric

In [ ]:
MODEL_VARIANTS = [
    "acf_net-base",
    "acf_net-base w/o softmax routing",
]


def select_variant_logits(outputs, model_variant):
    if model_variant == "acf_net-base":
        return outputs["logits_fused"]

    if model_variant == "acf_net-base w/o softmax routing":
        return (
            outputs["logits_expert_1"]
            + outputs["logits_expert_2"]
            + outputs["logits_expert_3"]
        ) / 3.0

    raise ValueError(f"Unknown model_variant: {model_variant}")


@torch.no_grad()
def evaluate_loader_brats_regions(model, loader, eval_cfg, split_name):
    """
    Evaluates both:
    - acf_net-base region-level softmax routing
    - no-softmax-routing comparison

    Returns one per-case row per model variant.
    """
    model.eval()
    rows = []

    print("=" * 80)
    print(f"Evaluating {split_name} set with BraTS WT/TC/ET metrics")
    print("=" * 80)
    print(f"Cases: {len(loader.dataset)}")

    if len(loader.dataset) == 0:
        print(f"[Warning] {split_name} loader has zero cases.")
        return pd.DataFrame(rows)

    for batch in tqdm(loader, desc=f"{split_name} BraTS-region evaluation", leave=True):
        case_id = extract_case_id(batch)
        spacing = batch_spacing_to_tuple(batch)

        image = batch["image"].to(eval_cfg.device, non_blocking=True)
        priors = batch["priors"].to(eval_cfg.device, non_blocking=True)
        region = batch["region"].to(eval_cfg.device, non_blocking=True)
        target = batch["seg"].to(eval_cfg.device, non_blocking=True)
        has_label = bool(batch.get("has_label", torch.tensor([True]))[0].item())

        outputs = model(image, priors, region)

        for model_variant in MODEL_VARIANTS:
            logits = select_variant_logits(outputs, model_variant)
            probs = torch.softmax(logits, dim=1)
            pred = torch.argmax(logits, dim=1)

            pred_np = pred[0].detach().cpu().numpy().astype(np.int16)
            probs_np = probs[0].detach().cpu().numpy().astype(np.float32)

            row = {
                "split": split_name,
                "case_id": case_id,
                "model_variant": model_variant,
                "has_label": has_label,
                "spacing_x": spacing[0],
                "spacing_y": spacing[1],
                "spacing_z": spacing[2],
                "dice_class_1": float("nan"),
                "dice_class_2": float("nan"),
                "dice_class_3": float("nan"),
                "mean_foreground_class_dice": float("nan"),
                "dice_wt": float("nan"),
                "dice_tc": float("nan"),
                "dice_et": float("nan"),
                "mean_region_dice": float("nan"),
                "hd95_wt": float("nan"),
                "hd95_tc": float("nan"),
                "hd95_et": float("nan"),
                "mean_hd95": float("nan"),
                "auroc_wt": float("nan"),
                "auprc_wt": float("nan"),
                "precision_wt": float("nan"),
                "recall_wt": float("nan"),
                "tp_wt": float("nan"),
                "fp_wt": float("nan"),
                "fn_wt": float("nan"),
                "pred_wt_voxels": int((pred_np > 0).sum()),
                "target_wt_voxels": float("nan"),
                "auroc_auprc_status": "label_missing",
            }

            if has_label:
                target_np = target[0].detach().cpu().numpy().astype(np.int16)

                row.update(classwise_dice(pred_np, target_np, num_classes=eval_cfg.num_classes))
                row.update(brats_region_metrics(pred_np, target_np, spacing=spacing, prefix=""))
                row.update(binary_wt_metrics(pred_np, target_np, probs_np))

            rows.append(row)

        del outputs, logits, probs, pred
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows)

    print(f"[{split_name}] rows: {len(df)}")
    if len(df):
        print(f"[{split_name}] cases with labels: {int(df.groupby('case_id')['has_label'].first().sum())}")

    return df

In [ ]:
def summarize_variant_metrics(df, split_name, model_variant):
    sub = df[(df["split"] == split_name) & (df["model_variant"] == model_variant)].copy()

    if len(sub) == 0:
        return {
            f"{split_name}_dice_wt": float("nan"),
            f"{split_name}_dice_tc": float("nan"),
            f"{split_name}_dice_et": float("nan"),
            f"{split_name}_mean_region_dice": float("nan"),
            f"{split_name}_hd95_wt": float("nan"),
            f"{split_name}_hd95_tc": float("nan"),
            f"{split_name}_hd95_et": float("nan"),
            f"{split_name}_mean_hd95": float("nan"),
            f"{split_name}_auroc_wt": float("nan"),
            f"{split_name}_auprc_wt": float("nan"),
            f"{split_name}_precision_wt": float("nan"),
            f"{split_name}_recall_wt": float("nan"),
            f"{split_name}_mean_foreground_class_dice": float("nan"),
            f"{split_name}_dice_class_1": float("nan"),
            f"{split_name}_dice_class_2": float("nan"),
            f"{split_name}_dice_class_3": float("nan"),
        }

    return {
        f"{split_name}_dice_wt": nanmean(sub["dice_wt"].values),
        f"{split_name}_dice_tc": nanmean(sub["dice_tc"].values),
        f"{split_name}_dice_et": nanmean(sub["dice_et"].values),
        f"{split_name}_mean_region_dice": nanmean(sub["mean_region_dice"].values),
        f"{split_name}_hd95_wt": nanmean(sub["hd95_wt"].values),
        f"{split_name}_hd95_tc": nanmean(sub["hd95_tc"].values),
        f"{split_name}_hd95_et": nanmean(sub["hd95_et"].values),
        f"{split_name}_mean_hd95": nanmean(sub["mean_hd95"].values),
        f"{split_name}_auroc_wt": nanmean(sub["auroc_wt"].values),
        f"{split_name}_auprc_wt": nanmean(sub["auprc_wt"].values),
        f"{split_name}_precision_wt": nanmean(sub["precision_wt"].values),
        f"{split_name}_recall_wt": nanmean(sub["recall_wt"].values),
        f"{split_name}_mean_foreground_class_dice": nanmean(sub["mean_foreground_class_dice"].values),
        f"{split_name}_dice_class_1": nanmean(sub["dice_class_1"].values),
        f"{split_name}_dice_class_2": nanmean(sub["dice_class_2"].values),
        f"{split_name}_dice_class_3": nanmean(sub["dice_class_3"].values),
    }


def build_ablation_summary_table(val_df, test_df, best_epoch, best_metric, selected_test_root):
    rows = []

    for model_variant in MODEL_VARIANTS:
        row = {
            "model_variant": model_variant,
            "best_epoch": int(best_epoch),
            "mean_val_dice_checkpoint_best_metric": safe_float(best_metric),
            "test_fastsurfer_root_used": selected_test_root,
        }

        row.update(summarize_variant_metrics(val_df, "val", model_variant))
        row.update(summarize_variant_metrics(test_df, "test", model_variant))

        rows.append(row)

    return pd.DataFrame(rows)


def build_single_model_summary_table(ablation_df):
    normal = ablation_df[ablation_df["model_variant"] == "acf_net-base"].copy()
    if len(normal) == 0:
        return pd.DataFrame()

    row = normal.iloc[0].to_dict()

    summary = {
        "model": "acf_net-base",
        "best_epoch": row.get("best_epoch", -1),
        "mean_val_dice": row.get("mean_val_dice_checkpoint_best_metric", float("nan")),
        "val_dice_wt": row.get("val_dice_wt", float("nan")),
        "val_dice_tc": row.get("val_dice_tc", float("nan")),
        "val_dice_et": row.get("val_dice_et", float("nan")),
        "val_mean_region_dice": row.get("val_mean_region_dice", float("nan")),
        "val_hd95_wt": row.get("val_hd95_wt", float("nan")),
        "val_hd95_tc": row.get("val_hd95_tc", float("nan")),
        "val_hd95_et": row.get("val_hd95_et", float("nan")),
        "val_mean_hd95": row.get("val_mean_hd95", float("nan")),
        "test_auroc_wt": row.get("test_auroc_wt", float("nan")),
        "test_auprc_wt": row.get("test_auprc_wt", float("nan")),
        "test_dice_wt": row.get("test_dice_wt", float("nan")),
        "test_dice_tc": row.get("test_dice_tc", float("nan")),
        "test_dice_et": row.get("test_dice_et", float("nan")),
        "test_mean_region_dice": row.get("test_mean_region_dice", float("nan")),
        "test_precision_wt": row.get("test_precision_wt", float("nan")),
        "test_recall_wt": row.get("test_recall_wt", float("nan")),
        "test_hd95_wt": row.get("test_hd95_wt", float("nan")),
        "test_hd95_tc": row.get("test_hd95_tc", float("nan")),
        "test_hd95_et": row.get("test_hd95_et", float("nan")),
        "test_mean_hd95": row.get("test_mean_hd95", float("nan")),
        "test_fastsurfer_root_used": row.get("test_fastsurfer_root_used", ""),
        "hd95_definition": "HD95; uses NIfTI spacing when available, otherwise voxel spacing",
        "validation_definition": "mean_val_dice is ckpt['best_metric']; WT/TC/ET validation metrics are recomputed from best_model.pt",
        "test_definition": "held-out splits['test'] evaluated after training",
    }

    return pd.DataFrame([summary])


def save_brats_region_results(eval_cfg, val_df, test_df, ablation_df, summary_df, missing_test, selected_test_root):
    ensure_dir(eval_cfg.results_dir)

    paths = {
        "summary_csv": os.path.join(eval_cfg.results_dir, "summary_results_acf_net_base_brats_regions.csv"),
        "summary_json": os.path.join(eval_cfg.results_dir, "summary_results_acf_net_base_brats_regions.json"),
        "val_csv": os.path.join(eval_cfg.results_dir, "per_case_val_results_acf_net_base_brats_regions.csv"),
        "test_csv": os.path.join(eval_cfg.results_dir, "per_case_test_results_acf_net_base_brats_regions.csv"),
        "ablation_csv": os.path.join(eval_cfg.results_dir, "ablation_results_acf_net_base_brats_regions.csv"),
        "ablation_json": os.path.join(eval_cfg.results_dir, "ablation_results_acf_net_base_brats_regions.json"),
    }

    val_df.to_csv(paths["val_csv"], index=False)
    test_df.to_csv(paths["test_csv"], index=False)
    ablation_df.to_csv(paths["ablation_csv"], index=False)
    summary_df.to_csv(paths["summary_csv"], index=False)

    summary_payload = {
        "summary": summary_df.to_dict(orient="records"),
        "missing_test_fastsurfer_cases": missing_test,
        "selected_test_fastsurfer_root": selected_test_root,
        "brats_regions": {
            "WT": [1, 2, 3],
            "TC": [1, 3],
            "ET": [3],
        },
        "hd95_rule": {
            "gt_empty_pred_empty": "Dice=1.0, HD95=0.0",
            "gt_nonempty_pred_empty": "Dice=0.0, HD95=NaN",
            "gt_empty_pred_nonempty": "Dice=0.0, HD95=NaN",
            "mean_hd95": "np.nanmean",
        },
    }

    with open(paths["summary_json"], "w") as f:
        json.dump(summary_payload, f, indent=2)

    with open(paths["ablation_json"], "w") as f:
        json.dump(
            {
                "ablation_results": ablation_df.to_dict(orient="records"),
                "uniform_fusion_definition": "(logits_expert_1 + logits_expert_2 + logits_expert_3) / 3",
                "base_definition": "outputs['logits_fused'] from learned region-level softmax routing",
            },
            f,
            indent=2,
        )

    print("=" * 80)
    print("Saved BraTS-region result files")
    print("=" * 80)
    for key, path in paths.items():
        print(f"{key}: {path}")

    return paths

In [ ]:
print("=" * 80)
print("Running acf_net-base BraTS-region evaluation + no-softmax-routing comparison")
print("=" * 80)

assert eval_cfg.best_checkpoint_path == "FILL IN WITH DIRECTORY FOR BEST MODEL CHECKPOINT/best_model.pt"
assert "FILL IN WITH DIRECTORY FOR ACF NET BASE BRATS RUNS/" in eval_cfg.best_checkpoint_path
assert os.path.exists(eval_cfg.best_checkpoint_path)

val_loader_eval, test_loader_eval, missing_test_cases, selected_test_root = build_brats_region_eval_loaders(eval_cfg)

acr_model_eval, acr_ckpt, best_epoch, best_metric = load_best_acr_model(eval_cfg)

print("\n[Validation evaluation]")
val_region_df = evaluate_loader_brats_regions(
    model=acr_model_eval,
    loader=val_loader_eval,
    eval_cfg=eval_cfg,
    split_name="val",
)

print("\n[Test evaluation]")
test_region_df = evaluate_loader_brats_regions(
    model=acr_model_eval,
    loader=test_loader_eval,
    eval_cfg=eval_cfg,
    split_name="test",
)

print("\n[Build summary and ablation tables]")
ablation_results_df = build_ablation_summary_table(
    val_df=val_region_df,
    test_df=test_region_df,
    best_epoch=best_epoch,
    best_metric=best_metric,
    selected_test_root=selected_test_root,
)

summary_results_df = build_single_model_summary_table(ablation_results_df)

print("=" * 80)
print("Final summary table: acf_net-base")
print("=" * 80)
display(summary_results_df)

print("=" * 80)
print("Ablation table: acf_net-base vs acf_net-base w/o softmax routing")
print("=" * 80)
display(ablation_results_df)

saved_paths = save_brats_region_results(
    eval_cfg=eval_cfg,
    val_df=val_region_df,
    test_df=test_region_df,
    ablation_df=ablation_results_df,
    summary_df=summary_results_df,
    missing_test=missing_test_cases,
    selected_test_root=selected_test_root,
)

print("\nDone.")

In [ ]:
print("=" * 80)
print("Running acf_net-base BraTS-region evaluation + no-softmax-routing comparison")
print("=" * 80)

assert eval_cfg.best_checkpoint_path == "FILL IN WITH DIRECTORY FOR BEST MODEL CHECKPOINT/best_model.pt"
assert "FILL IN WITH DIRECTORY FOR ACF NET BASE BRATS RUNS/" in eval_cfg.best_checkpoint_path
assert os.path.exists(eval_cfg.best_checkpoint_path)

val_loader_eval, test_loader_eval, missing_test_cases, selected_test_root = build_brats_region_eval_loaders(eval_cfg)

acr_model_eval, acr_ckpt, best_epoch, best_metric = load_best_acr_model(eval_cfg)

print("\n[Validation evaluation]")
val_region_df = evaluate_loader_brats_regions(
    model=acr_model_eval,
    loader=val_loader_eval,
    eval_cfg=eval_cfg,
    split_name="val",
)

print("\n[Test evaluation]")
test_region_df = evaluate_loader_brats_regions(
    model=acr_model_eval,
    loader=test_loader_eval,
    eval_cfg=eval_cfg,
    split_name="test",
)

print("\n[Build summary and ablation tables]")
ablation_results_df = build_ablation_summary_table(
    val_df=val_region_df,
    test_df=test_region_df,
    best_epoch=best_epoch,
    best_metric=best_metric,
    selected_test_root=selected_test_root,
)

summary_results_df = build_single_model_summary_table(ablation_results_df)

print("=" * 80)
print("Final summary table: acf_net-base")
print("=" * 80)
display(summary_results_df)

print("=" * 80)
print("Ablation table: acf_net-base vs acf_net-base w/o softmax routing")
print("=" * 80)
display(ablation_results_df)

saved_paths = save_brats_region_results(
    eval_cfg=eval_cfg,
    val_df=val_region_df,
    test_df=test_region_df,
    ablation_df=ablation_results_df,
    summary_df=summary_results_df,
    missing_test=missing_test_cases,
    selected_test_root=selected_test_root,
)

print("\nDone.")

In [ ]:
compact_cols = [
    "model_variant",
    "val_dice_wt",
    "val_dice_tc",
    "val_dice_et",
    "val_mean_region_dice",
    "test_dice_wt",
    "test_dice_tc",
    "test_dice_et",
    "test_mean_region_dice",
    "test_auroc_wt",
    "test_auprc_wt",
    "test_precision_wt",
    "test_recall_wt",
    "test_hd95_wt",
    "test_hd95_tc",
    "test_hd95_et",
    "test_mean_hd95",
]

compact_ablation_df = ablation_results_df[compact_cols].copy()

print("=" * 80)
print("Compact BraTS-region ablation table")
print("=" * 80)
display(compact_ablation_df)